In [1]:
import os

import random
import datetime
from collections import defaultdict

# !pip install networkx
# !pip install pandas
# !pip install scikit-learn
# !pip install scipy
# !pip install pyparsing
# !pip install fsspec
import networkx as nx
import multiprocessing as mp

import heapq
import time

#from google.colab import drive
#!pip install openpyxl
import pandas as pd



# === Mount Google Drive ===
#drive.mount('/content/drive', force_remount=True)

def read_graph(file_path):
    edges = []
    
    with open(file_path, 'r') as f:
        for line in f:
            # Strip whitespace and skip empty lines
            line = line.strip()
            if not line:
                continue
            
            # Skip comments ('c') and problem definition ('p') lines
            # We derive the nodes dynamically from the 'a' lines, 
            # so strict parsing of 'p' is not required for the topology.
            if line.startswith('c') or line.startswith('p'):
                continue
            
            # Parse arc lines: a <source> <target> <weight> <transit_time>
            if line.startswith('a'):
                parts = line.split()
                # parts[0] is 'a'
                # parts[1] is source
                # parts[2] is target
                # parts[3] is weight
                # parts[4] is transit_time (ignored)
                
                u = parts[1]
                v = parts[2]
                w = float(parts[3])
                
                edges.append((u, v, w))

    # The following logic remains identical to the original function
    # to ensure the return types (dictionaries, lists) are consistent.
    
    # Create a sorted set of unique nodes to establish a deterministic index mapping
    node_set = sorted(set(u for u, v, _ in edges).union(v for u, v, _ in edges))
    
    # Map node IDs (strings) to 0-based integers
    node_to_index = {node: i for i, node in enumerate(node_set)}
    index_to_node = {i: node for node, i in node_to_index.items()}
    
    # Rebuild edges using the internal 0-based indices
    edges_indexed = [(node_to_index[u], node_to_index[v], w) for (u, v, w) in edges]
    
    return edges_indexed, node_to_index, index_to_node
def load_initial_scores(csv_path, node_to_index):
    df = pd.read_csv(csv_path)
    df['Node ID'] = df['Node ID'].astype(str).str.strip()
    rank_map = {row['Node ID']: row['Order'] for _, row in df.iterrows()}

    scores = {}
    for node_str, idx in node_to_index.items():
        if node_str in rank_map:
            scores[idx] = int(rank_map[node_str])

    # Assign unique ranks to unranked nodes
    max_rank = max(scores.values(), default=0) + 1
    for node_str, idx in node_to_index.items():
        if idx not in scores:
            scores[idx] = max_rank
            max_rank += 1

    return scores

import random




def compute_forward_weight(edges, scores):
    return sum(w for u, v, w in edges if scores[u] < scores[v])

import datetime

def log_message(message, log_file_path):
    """
    Appends a message with a timestamp to the specified log file.
    """
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    formatted_message = f"[{timestamp}] {message}\n"
    
    # 'a' mode opens the file for appending; creates the file if it doesn't exist.
    with open(log_file_path, "a") as f:
        f.write(formatted_message)
    
    # Optional: Print to console as well so you can see progress in real-time
    #print(formatted_message.strip())

# Example usage within your workflow:
# log_message("SCC decomposition started.", log_path)
# log_message("Hybrid function iteration 1 complete.", log_path)



In [4]:
import pandas as pd
from collections import defaultdict

def csv_connectome_to_dimacs_d(
    csv_path: str,
    out_path: str = "connectome.d",
    write_transit_time: bool = True,
    transit_time_value: int = 1,
    chunksize: int | None = None,
):
    """
    Convert connectome_graph.csv -> connectome.d (DIMACS-like).

    Output lines:
      a <source> <target> <weight> <transit_time>

    - Aggregates parallel arcs by summing weights.
    - Writes node IDs as strings (no re-indexing needed; your DIMACS reader
      builds a deterministic mapping anyway).
    - Optionally includes a dummy transit_time field (default 1). The DIMACS reader ignores it.
    """

    agg = defaultdict(float)

    def process_df(df: pd.DataFrame):
        # normalize/rename columns to a stable schema
        df = df.rename(columns={
            "Source Node  ID": "source",
            "Target Node ID": "target",
            "Edge Weight": "weight",
        })

        # keep only needed columns
        df = df[["source", "target", "weight"]].copy()

        # string IDs (matches your csv reader behavior)
        df["source"] = df["source"].astype(str)
        df["target"] = df["target"].astype(str)

        # numeric weight; drop bad rows
        df["weight"] = pd.to_numeric(df["weight"], errors="coerce")
        df = df.dropna(subset=["weight"])

        # aggregate in Python dict (fast enough; also avoids pandas groupby memory spikes)
        for s, t, w in df.itertuples(index=False, name=None):
            agg[(s, t)] += float(w)

    if chunksize is None:
        df = pd.read_csv(csv_path)
        process_df(df)
    else:
        for df_chunk in pd.read_csv(csv_path, chunksize=chunksize):
            process_df(df_chunk)

    # Write DIMACS-like .d
    with open(out_path, "w") as f:
        f.write("c Converted from connectome_graph.csv\n")
        # (Optional) header line; your reader ignores 'p' lines anyway.
        f.write(f"p arc {len({n for uv in agg for n in uv})} {len(agg)}\n")

        # deterministic output order
        for (u, v) in sorted(agg.keys()):
            w = agg[(u, v)]
            if write_transit_time:
                f.write(f"a {u} {v} {w:.12g} {transit_time_value}\n")
            else:
                # still OK: your reader accepts extra fields, but it only *requires* 4 tokens;
                # if you omit transit_time, it will still read because len(parts) >= 4 holds.
                f.write(f"a {u} {v} {w:.12g}\n")

    return out_path


# Example usage:
csv_connectome_to_dimacs_d("/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome_graph.csv", "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d", chunksize=2_000_000)


'/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d'

In [2]:
def compare_backward_weights(edges, sbef, saft):
    """
    Given edges and two score dictionaries (sbef and saft),
    returns:
    - sum of backward edge weights in sbef that are NOT backward in saft (Improved)
    - sum of backward edge weights in saft that are NOT backward in sbef (Regressed)
    - sum of backward edges in both
    - sum of forward edges in both
    """
    sum_sbef_backward_only = 0.0
    sum_saft_backward_only = 0.0
    sum_both_backward = 0.0
    sum_both_forward = 0.0
    
    # Set to track edges that were backward ONLY in the 'before' state
    only_backward_in_before = set()

    for u, v, w in edges:
        # Check backward condition: score[u] > score[v]
        is_backward_sbef = sbef[u] > sbef[v]
        is_backward_saft = saft[u] > saft[v]

        if is_backward_sbef and not is_backward_saft:
            # It was backward, now it is forward (Improvement)
            sum_sbef_backward_only += w
            only_backward_in_before.add((u, v, w))
            
        elif is_backward_saft and not is_backward_sbef:
            # It was forward, now it is backward (Regression)
            sum_saft_backward_only += w
            
        elif is_backward_sbef and is_backward_saft:
            # Still backward
            sum_both_backward += w
            
        else:
            # Not backward in either (Forward in both)
            sum_both_forward += w

    return sum_sbef_backward_only, sum_saft_backward_only, sum_both_backward, sum_both_forward, only_backward_in_before

def all_scores_unique(scores):
    """
    Returns True if all nodes have unique scores, otherwise False.
    """
    values = list(scores.values())
    return len(values) == len(set(values))

def global_scc_topo_reorder_scores(G, edges, scores, log_path, debug=False, total_weight=None):
    """
    Reorder 'scores' so that each SCC's vertices are contiguous in rank and SCCs
    appear in a topological order of the condensation DAG.

    CRITERION: Backward Weight (BW) must NOT increase.

    Logging:
      - Uses ONLY log_message(message, log_path). No prints.

    BW computation:
      - If total_weight is provided, BW is computed as BW = total_weight - FW
        using compute_forward_weight (fast + consistent).
      - Otherwise, falls back to compute_backward_weight(edges, scores).
    """
    import networkx as nx

    # -------------------- Helpers: BW via total_weight - FW --------------------
    def _bw_of(scores_map):
        if total_weight is not None:
            fw = float(compute_forward_weight(edges, scores_map))
            return float(total_weight) - fw
        return float(compute_backward_weight(edges, scores_map))

    if debug:
        log_message("global_scc_topo_reorder_scores: running Global SCC topo reorder...", log_path)

    # 1) Identify SCCs
    sccs = list(nx.strongly_connected_components(G))
    if not sccs:
        # Empty graph: keep scores as-is
        if debug:
            log_message("global_scc_topo_reorder_scores: no SCCs found (empty graph). Returning original scores.", log_path)
        return scores.copy()

    # 2) Map nodes to SCC IDs
    node_to_scc = {}
    for scc_id, scc in enumerate(sccs):
        for n in scc:
            node_to_scc[n] = scc_id

    # 3) Build the Condensation DAG (SCC Graph)
    scc_dag = nx.DiGraph()
    scc_dag.add_nodes_from(range(len(sccs)))

    # Add edges between SCCs based on original edges
    for u, v, _w in edges:
        su = node_to_scc[u]
        sv = node_to_scc[v]
        if su != sv:
            scc_dag.add_edge(su, sv)

    # 4) Topological Sort of SCCs
    scc_order = list(nx.topological_sort(scc_dag))

    # 5) BW BEFORE reordering
    bw_before = _bw_of(scores)

    # 6) Generate New Scores:
    #    Preserve order inside SCC by old rank; order SCC blocks by topo order
    new_scores = {}
    next_rank = 0

    for scc_id in scc_order:
        nodes_in_scc = list(sccs[scc_id])
        nodes_in_scc_sorted = sorted(nodes_in_scc, key=lambda n: scores[n])
        for n in nodes_in_scc_sorted:
            new_scores[n] = next_rank
            next_rank += 1

    # 7) BW AFTER reordering
    bw_after = _bw_of(new_scores)

    # 8) Validation: BW must NOT increase
    if bw_after > bw_before + 1e-9:
        error_msg = (
            "❌ global_scc_topo_reorder_scores: Global SCC topo reorder INCREASED BW: "
            f"before={bw_before:.6f}, after={bw_after:.6f}"
        )
        log_message(error_msg, log_path)
        raise RuntimeError(error_msg)

    if debug:
        log_message(
            f"global_scc_topo_reorder_scores: done. BW_before={bw_before:.6f}, BW_after={bw_after:.6f} (non-increasing).",
            log_path
        )

    return new_scores


In [3]:
def apply_new_strategy(scores, u, v, between, out_edges, in_edges, edges_dict, log_path, debug=False):
    """
    Optimizes the position of u and v within the block [v] + between + [u].

    Notes on objective:
      - Increasing forward-edge weight by Δ implies decreasing backward-edge weight by the same Δ,
        because total edge weight is constant. So we treat the improvement value returned here as
        "backward-weight reduction" (BW reduction).
    """
    import time

    PROFILE_THRESHOLD = 0.08  # seconds
    t_total0 = time.time()

    # Scores are modified in-place during reorder step.
    # We only snapshot the ranks we need (do NOT copy full dict for performance).
    ranks_before = (scores[u], scores[v])

    # ----------------- 1) Define neighbors of v and u within 'between' -----------------
    # 'between' is assumed sorted by current rank, and list comprehension preserves that order.
    v_neighbors_sorted = [
        node for node in between
        if (node, v) in edges_dict or (v, node) in edges_dict
    ]
    u_neighbors_sorted = [
        node for node in between
        if (node, u) in edges_dict or (u, node) in edges_dict
    ]

    # ----------------- 2) Prefix/Suffix sums -----------------
    # Prefix sums over v_neighbors (from left to right)
    outgoing_v = [0.0] * (len(v_neighbors_sorted) + 1)
    ingoing_v = [0.0] * (len(v_neighbors_sorted) + 1)
    for i, node in enumerate(v_neighbors_sorted):
        outgoing_v[i + 1] = outgoing_v[i] + edges_dict.get((v, node), 0.0)
        ingoing_v[i + 1] = ingoing_v[i] + edges_dict.get((node, v), 0.0)

    # Suffix sums over u_neighbors (from right to left)
    outgoing_u = [0.0] * (len(u_neighbors_sorted) + 1)
    ingoing_u = [0.0] * (len(u_neighbors_sorted) + 1)
    for i in reversed(range(len(u_neighbors_sorted))):
        node_i = u_neighbors_sorted[i]
        outgoing_u[i] = outgoing_u[i + 1] + edges_dict.get((u, node_i), 0.0)
        ingoing_u[i] = ingoing_u[i + 1] + edges_dict.get((node_i, u), 0.0)

    # ----------------- 3) Two-pointer sweep to find best cut -----------------
    best_bw_reduction = float("-inf")
    best_y = -1
    best_j = -1

    w_uv = edges_dict.get((u, v), 0.0)
    w_vu = edges_dict.get((v, u), 0.0)

    # ΔFW for swapping u/v relative to each other; equals BW reduction by same amount.
    base_bw_reduction = w_uv - w_vu

    u_scores_list = [scores[node] for node in u_neighbors_sorted]
    len_u = len(u_scores_list)
    j = 0

    for i, nv in enumerate(v_neighbors_sorted):
        s = scores[nv]

        # Advance j to maintain rank monotonicity
        while j < len_u and u_scores_list[j] <= s:
            j += 1

        if j < len_u:
            # BW reduction = ΔFW (because total weight is constant)
            bw_reduction = (
                base_bw_reduction
                + ingoing_v[i + 1] - outgoing_v[i + 1]
                - ingoing_u[j] + outgoing_u[j]
            )

            if bw_reduction > best_bw_reduction:
                best_bw_reduction = bw_reduction
                best_y = i
                best_j = j

    # ----------------- 4) Early exit if no improvement -----------------
    if best_bw_reduction <= 0:
        ranks_after = (scores[u], scores[v])
        return False, 0.0, ranks_before, ranks_after, {}

    # ----------------- 5) Apply reordering (success case) -----------------
    # best_y is index in v_neighbors_sorted; map to index in 'between'
    split_node = v_neighbors_sorted[best_y]
    split_index = between.index(split_node)

    left = between[:split_index + 1]
    right = between[split_index + 1:]
    new_order = left + [u, v] + right

    # Assign new scores sequentially starting from the original score of v
    base_score = scores[v]
    for k, node in enumerate(new_order):
        scores[node] = base_score + k

    ranks_after = (scores[u], scores[v])

    # ----------------- 6) Profiling Log -----------------
    t_total = time.time() - t_total0
    if t_total > PROFILE_THRESHOLD:
        log_message(
            f"SLOW success: u={u}, v={v}, |between|={len(between)}, "
            f"bw_reduction={best_bw_reduction:.3f}, total_time={t_total:.4f}s",
            log_path
        )

    return True, best_bw_reduction, ranks_before, ranks_after, {}


In [4]:
def compute_all_gains_local(scores, u, v, out_edges, in_edges):
    idx_u = scores[u]
    idx_v = scores[v]
    if idx_u <= idx_v:
        raise ValueError(f"Invalid edge ({u}->{v}) is already forward!")

    lo, hi = idx_v, idx_u

    # Local bindings for speed
    scores_local = scores
    idx_u_local = idx_u
    idx_v_local = idx_v

    def new_rank(node, rank_node, strategy):
        """Compute new rank for 'node' under given strategy, using its original rank."""
        if strategy == 'swap':
            if node == u:
                return idx_v_local
            if node == v:
                return idx_u_local
            return rank_node

        elif strategy == 'mvvafu':  # move v after u
            if node == v:
                return idx_u_local
            if idx_v_local < rank_node <= idx_u_local:
                return rank_node - 1
            return rank_node

        elif strategy == 'mvubfv':  # move u before v
            if node == u:
                return idx_v_local
            if idx_v_local <= rank_node < idx_u_local:
                return rank_node + 1
            return rank_node

        # Should never happen
        return rank_node

    # Store just the gains, we don’t actually use the f2b/b2f breakdown externally
    swap_gain = 0.0
    mvvafu_gain = 0.0
    mvubfv_gain = 0.0

    # If you ever care again:
    # swap_f2b = swap_b2f = mvvafu_f2b = mvvafu_b2f = mvubfv_f2b = mvubfv_b2f = 0.0

    edges_checked = 0

    def process_edge(a, b, w):
        nonlocal edges_checked
        nonlocal swap_gain, mvvafu_gain, mvubfv_gain

        sa = scores_local[a]
        sb = scores_local[b]

        # Only edges touching [lo, hi] matter
        if not (lo <= sa <= hi or lo <= sb <= hi):
            return

        edges_checked += 1
        old_forward = (sa < sb)

        # ---- swap ----
        ra2 = new_rank(a, sa, 'swap')
        rb2 = new_rank(b, sb, 'swap')
        new_forward = (ra2 < rb2)
        if old_forward and not new_forward:
            swap_gain -= w
            # swap_f2b += w
        elif (not old_forward) and new_forward:
            swap_gain += w
            # swap_b2f += w

        # ---- mvvafu ----
        ra2 = new_rank(a, sa, 'mvvafu')
        rb2 = new_rank(b, sb, 'mvvafu')
        new_forward = (ra2 < rb2)
        if old_forward and not new_forward:
            mvvafu_gain -= w
            # mvvafu_f2b += w
        elif (not old_forward) and new_forward:
            mvvafu_gain += w
            # mvvafu_b2f += w

        # ---- mvubfv ----
        ra2 = new_rank(a, sa, 'mvubfv')
        rb2 = new_rank(b, sb, 'mvubfv')
        new_forward = (ra2 < rb2)
        if old_forward and not new_forward:
            mvubfv_gain -= w
            # mvubfv_f2b += w
        elif (not old_forward) and new_forward:
            mvubfv_gain += w
            # mvubfv_b2f += w

    # Collect candidate edges (neighbors of u or v within [lo, hi])
    # Using a set is still OK here to avoid double-counting; but we only
    # hash (a,b,w) once per neighbor.
    edges_to_check = set()

    # Local bindings for adjacency to avoid global lookups
    out_u = out_edges[u]
    in_u  = in_edges[u]
    out_v = out_edges[v]
    in_v  = in_edges[v]
    scores_vals = scores_local  # alias

    for x, w in out_u:
        rx = scores_vals[x]
        if lo <= rx <= hi:
            edges_to_check.add((u, x, w))

    for x, w in in_u:
        rx = scores_vals[x]
        if lo <= rx <= hi:
            edges_to_check.add((x, u, w))

    for x, w in out_v:
        rx = scores_vals[x]
        if lo <= rx <= hi:
            edges_to_check.add((v, x, w))

    for x, w in in_v:
        rx = scores_vals[x]
        if lo <= rx <= hi:
            edges_to_check.add((x, v, w))

    for a, b, w in edges_to_check:
        process_edge(a, b, w)

    return swap_gain, mvvafu_gain, mvubfv_gain


In [5]:
def select_nonconflicting_edge_indices_dp(backward_edges):
    """
    Given a list of backward_edges = (u, v, w, lo, hi) with lo < hi (ranks),
    select a maximum-size subset of edges whose [lo, hi] intervals do not overlap.

    IMPORTANT:
      - This selects a maximum-cardinality set (not weight-based). The tuple
        contains w only because callers often carry it around for later
        backward-weight accounting, but it is NOT used here.
      - We compress coordinates so DP is defined only on ranks that are
        endpoints of some edge (lo or hi), not on [0..max_rank].

    Returns:
      selected_indices: set of LOCAL indices (0..len(backward_edges)-1)
                        of chosen edges.

    Complexity per call:
      O(E log E) dominated by sorting endpoints (≤ 2E) and sorting edges by hi.
      DP itself is O(E + K), K ≤ 2E.
    """
    if not backward_edges:
        return set()

    # 1) Collect all distinct endpoints (lo, hi)
    endpoints = set()
    for (_, _, _, lo, hi) in backward_edges:
        endpoints.add(lo)
        endpoints.add(hi)

    sorted_endpoints = sorted(endpoints)
    coord_index = {rank: i for i, rank in enumerate(sorted_endpoints)}
    K = len(sorted_endpoints)

    # 2) Convert edges to compressed coordinates and sort by hi_pos
    #    We keep local index so we can return indices into backward_edges.
    edges = []
    for idx, (u, v, w, lo, hi) in enumerate(backward_edges):
        lo_pos = coord_index[lo]
        hi_pos = coord_index[hi]
        edges.append((hi_pos, lo_pos, idx))

    # Sort by end coordinate
    edges.sort(key=lambda x: x[0])

    # Helper: for each edge in sorted order, find the last compatible edge index
    # (i.e., previous edge with hi_pos < current lo_pos). We use binary search on hi_pos list.
    hi_list = [e[0] for e in edges]

    def _rightmost_compatible_edge_index(lo_pos):
        # Return largest j such that hi_list[j] < lo_pos, or -1 if none.
        # (strict < avoids overlap at a point; if you want allow touching, change to <=)
        import bisect
        j = bisect.bisect_left(hi_list, lo_pos) - 1
        return j

    # 3) Weighted-interval-scheduling DP but with weight=1 for every edge
    n = len(edges)
    dp = [0] * (n + 1)         # dp[i] = best using first i edges (edges[0..i-1])
    take = [False] * (n + 1)   # whether we take edge i-1

    # Precompute p(i): index of last compatible edge for edge i-1 (0-based in edges)
    p = [-1] * n
    for i in range(n):
        hi_pos, lo_pos, _ = edges[i]
        p[i] = _rightmost_compatible_edge_index(lo_pos)

    for i in range(1, n + 1):
        # Option 1: skip edge i-1
        opt1 = dp[i - 1]

        # Option 2: take edge i-1
        # then we add 1 + dp[p(i-1)+1]
        j = p[i - 1]
        opt2 = 1 + (dp[j + 1] if j >= 0 else 0)

        if opt2 > opt1:
            dp[i] = opt2
            take[i] = True
        else:
            dp[i] = opt1
            take[i] = False

    # 4) Reconstruct chosen edges (original local indices into backward_edges)
    selected_indices = set()
    i = n
    while i > 0:
        if take[i]:
            hi_pos, lo_pos, local_idx = edges[i - 1]
            selected_indices.add(local_idx)
            j = p[i - 1]
            i = j + 1
        else:
            i -= 1

    return selected_indices


In [6]:
def worker_loop(worker_idx, num_procs,
                backward_edges,
                scores_snapshot,
                out_edges, in_edges, edges_dict,
                active_intervals,   # unused now
                used_intervals,     # unused now
                lock,               # unused now
                edge_queue,
                result_queue,
                log_path):
    """
    Worker processes backward_edges indices from edge_queue, tries:
      1) apply_new_strategy (returns BW reduction)
      2) greedy fallback (assumed to return BW reduction; if it returns FW gain, it's identical)

    Logging:
      - MUST use only log_message(..., log_path)
      - No other printing/logging is used here.
    """
    import time

    # Thresholds for profiling logs (seconds)
    BETWEEN_WARN = 0.05
    APPLY_WARN = 0.10
    LOCK_FRACTION_WARN = 0.25  # lock is unused now; kept for compatibility
    BETWEEN_SAFETY_CHECK_LIMIT = 5
    between_checks_done = 0

    # -------------------- SNAPSHOT / RANK ARRAY --------------------
    max_rank = max(scores_snapshot.values())
    rank_to_node = [None] * (max_rank + 1)

    for node, r in scores_snapshot.items():
        if r < 0 or r > max_rank:
            log_message(f"RUNTIME ERROR: Invalid rank in scores_snapshot: node={node}, rank={r}", log_path)
            raise RuntimeError("Invalid rank in scores_snapshot.")
        if rank_to_node[r] is not None:
            log_message(
                f"RUNTIME ERROR: Duplicate rank in scores_snapshot: rank={r}, "
                f"nodes={rank_to_node[r]} and {node}",
                log_path
            )
            raise RuntimeError("Non-injective ranks in scores_snapshot.")
        rank_to_node[r] = node

    # Quick sanity: ensure used ranks map to non-None nodes
    for r in set(scores_snapshot.values()):
        if rank_to_node[r] is None:
            log_message(f"RUNTIME ERROR: rank_to_node[{r}] is None while some node has rank {r}", log_path)
            raise RuntimeError("rank_to_node inconsistent with scores_snapshot.")

    worker_start = time.time()

    total_lock_time = 0.0  # lock is unused now, kept for summary compatibility
    total_apply_strategy_time = 0.0
    total_greedy_time = 0.0
    total_queue_wait_time = 0.0
    total_edge_processing_time = 0.0

    edges_processed = 0
    successes = 0
    failures = 0
    extended_successes = 0
    greedy_successes = 0

    scores_base = scores_snapshot
    out_edges_base = out_edges
    in_edges_base = in_edges
    edges_array = backward_edges

    # Helper: call compute_all_gains_local with/without log_path, depending on its signature.
    def _compute_gains(scores_base_local, u0, v0, out_e, in_e, log_file_path):
        try:
            # Preferred: allow function to log using log_message via passed path, if it supports it.
            return compute_all_gains_local(scores_base_local, u0, v0, out_e, in_e, log_file_path)
        except TypeError:
            # Backward compatible: original signature without log_path
            return compute_all_gains_local(scores_base_local, u0, v0, out_e, in_e)

    while True:
        # -----------------------------------------------
        # 1) POP EDGE INDEX FROM QUEUE (non-blocking)
        # -----------------------------------------------
        t0 = time.time()
        try:
            idx = edge_queue.get_nowait()
        except Exception:
            total_queue_wait_time += (time.time() - t0)
            break
        total_queue_wait_time += (time.time() - t0)

        edge_start = time.time()
        edges_processed += 1

        # Default values for safe exception handling
        u = v = None
        lo = hi = None
        mode = "none"
        success = False
        delta_bw_reduction = 0.0
        changed_scores = {}
        t_between = 0.0
        t_apply_dur = 0.0
        greedy_dur = 0.0

        try:
            u, v, w, lo, hi = edges_array[idx]

            idx_u = scores_base[u]
            idx_v = scores_base[v]

            # This worker only acts on edges that are backward in this snapshot
            # (u appears after v -> idx_u > idx_v).
            if not (idx_u > idx_v):
                failures += 1
                continue

            # -----------------------------------------------
            # Build "between" using rank_to_node slice
            # -----------------------------------------------
            t_btw = time.time()
            if idx_u - idx_v > 1:
                slice_raw = rank_to_node[idx_v + 1: idx_u]
                between = [node for node in slice_raw if node is not None]
            else:
                between = []
            t_between = time.time() - t_btw

            if t_between > BETWEEN_WARN:
                log_message(f"'between' build slow: len={len(between)} time={t_between:.6f}s", log_path)

            # Optional safety check vs slow method (first few calls)
            if between_checks_done < BETWEEN_SAFETY_CHECK_LIMIT:
                between_checks_done += 1
                slow_between = [node for node, r in scores_base.items() if idx_v < r < idx_u]
                if set(between) != set(slow_between):
                    log_message("RUNTIME ERROR: Fast 'between' does not match slow method.", log_path)
                    raise RuntimeError("Fast 'between' implementation mismatch detected.")

            # Local copy for this edge attempt
            scores = scores_base.copy()

            # -----------------------------------------------
            # 2) Extended strategy (returns BW reduction)
            # -----------------------------------------------
            t_apply = time.time()
            try:
                ext_success, ext_bw_reduction, rB, rA, dbg = apply_new_strategy(
                    scores, u, v, between,
                    out_edges_base, in_edges_base, edges_dict,
                    log_path,
                    debug=False
                )
                success = bool(ext_success)
                delta_bw_reduction = float(ext_bw_reduction)
                mode = "extended"
            except Exception as e:
                log_message(f"ERROR apply_new_strategy({u}->{v}): {repr(e)}", log_path)
                success = False
                delta_bw_reduction = 0.0
                mode = "extended_error"

            t_apply_dur = time.time() - t_apply
            total_apply_strategy_time += t_apply_dur

            if t_apply_dur > APPLY_WARN:
                log_message(f"apply_new_strategy slow for edge ({u}->{v}): {t_apply_dur:.6f}s", log_path)

            if success and delta_bw_reduction <= 0.0:
                log_message(
                    f"RUNTIME WARNING: success=True but bw_reduction={delta_bw_reduction:.6f} for ({u}->{v}). "
                    f"Treating as failure.",
                    log_path
                )
                success = False
                delta_bw_reduction = 0.0
                mode = "extended_nonpositive_delta"

            # -----------------------------------------------
            # 3) Greedy fallback
            # -----------------------------------------------
            if not (success and delta_bw_reduction > 0.0):
                t_g = time.time()
                g1, g2, g3 = _compute_gains(scores_base, u, v, out_edges_base, in_edges_base, log_path)
                greedy_dur = time.time() - t_g
                total_greedy_time += greedy_dur

                best_gain = max(g1, g2, g3)
                if best_gain > 0:
                    scores = scores_base.copy()
                    delta_bw_reduction = float(best_gain)
                    success = True

                    if best_gain == g1:
                        # swap u and v
                        scores[u], scores[v] = idx_v, idx_u
                        mode = "greedy/swap"
                    elif best_gain == g2:
                        # move v after u
                        for k, r in scores.items():
                            if idx_v < r <= idx_u:
                                scores[k] = r - 1
                        scores[v] = idx_u
                        mode = "greedy/move_v_after_u"
                    else:
                        # move u before v
                        for k, r in scores.items():
                            if idx_v <= r < idx_u:
                                scores[k] = r + 1
                        scores[u] = idx_v
                        mode = "greedy/move_u_before_v"
                else:
                    success = False
                    delta_bw_reduction = 0.0
                    mode = "none" if mode != "extended_error" else "none_after_error"

            # -----------------------------------------------
            # 4) Collect rank changes within [lo, hi]
            # -----------------------------------------------
            if success and delta_bw_reduction > 0.0:
                for node, r_old in scores_base.items():
                    if lo <= r_old <= hi:
                        r_new = scores[node]
                        if r_new != r_old:
                            changed_scores[node] = r_new

                if not changed_scores:
                    log_message(
                        f"RUNTIME ERROR: success with positive bw_reduction but no changed_scores for edge ({u}->{v})",
                        log_path
                    )
                    success = False
                    delta_bw_reduction = 0.0
                    mode = f"{mode}_no_changes"

                # Validate: any changed node must have old rank inside [lo,hi]
                for node in changed_scores.keys():
                    old_r = scores_base[node]
                    if not (lo <= old_r <= hi):
                        log_message(
                            f"RUNTIME ERROR: Node {node} changed but old rank {old_r} outside interval [{lo},{hi}]",
                            log_path
                        )
                        raise RuntimeError("Changed rank outside edge interval.")

            # -----------------------------------------------
            # 5) Accounting + result send
            # -----------------------------------------------
            if success and delta_bw_reduction > 0.0:
                successes += 1
                if mode.startswith("extended"):
                    extended_successes += 1
                elif mode.startswith("greedy"):
                    greedy_successes += 1

                # log_message(
                #     f"SUCCESS ({u}->{v}) BW_Reduct={delta_bw_reduction:.4f} mode={mode} "
                #     f"interval=({lo},{hi}) changed={len(changed_scores)}",
                #     log_path
                # )
            else:
                failures += 1

            edge_total_time = time.time() - edge_start
            total_edge_processing_time += edge_total_time

            result_queue.put({
                "worker_idx": worker_idx,
                "u": u,
                "v": v,
                "success": bool(success and delta_bw_reduction > 0.0),
                "delta": float(delta_bw_reduction),   # BW reduction
                "changed_scores": changed_scores,
                "mode": mode,
                "interval": (lo, hi),
                "timing": {
                    "apply_new_strategy": float(t_apply_dur),
                    "greedy_time": float(greedy_dur),
                    "between_list_time": float(t_between),
                    "edge_total_time": float(edge_total_time),
                },
            })

        except Exception as e:
            failures += 1
            edge_total_time = time.time() - edge_start
            total_edge_processing_time += edge_total_time

            log_message(
                f"UNHANDLED ERROR processing edge index {idx} ({u}->{v}): {repr(e)}",
                log_path
            )

            result_queue.put({
                "worker_idx": worker_idx,
                "u": u,
                "v": v,
                "success": False,
                "delta": 0.0,
                "changed_scores": {},
                "mode": "error",
                "interval": (lo, hi),
                "timing": {
                    "apply_new_strategy": 0.0,
                    "greedy_time": 0.0,
                    "between_list_time": 0.0,
                    "edge_total_time": float(edge_total_time),
                },
            })

    # -----------------------------------------------
    # Worker summary
    # -----------------------------------------------
    runtime = time.time() - worker_start
    lock_fraction = (total_lock_time / runtime) if runtime > 0 else 0.0

    if lock_fraction > LOCK_FRACTION_WARN:
        log_message(f"WARNING: High lock contention: {lock_fraction * 100:.2f}% of time", log_path)

    result_queue.put({
        "worker_idx": worker_idx,
        "done": True,
        "edges_processed": edges_processed,
        "successes": successes,
        "extended_successes": extended_successes,
        "greedy_successes": greedy_successes,
        "failures": failures,
        "runtime": float(runtime),
        "timing_totals": {
            "queue_wait": float(total_queue_wait_time),
            "lock_time": float(total_lock_time),
            "apply_total": float(total_apply_strategy_time),
            "greedy_total": float(total_greedy_time),
            "edge_processing_total": float(total_edge_processing_time),
        },
    })


In [7]:
# Global variables for worker processes
G_GLOBAL = None
EDGES_GLOBAL = None
SCORES_BEFORE_GLOBAL = None

def _init_refine_worker(G, edges, scores_before):
    """
    Initializer for multiprocessing workers.
    Each worker gets its own copy of:
      - G (networkx DiGraph)
      - edges (list of edges)
      - scores_before (dict of scores)
    """
    global G_GLOBAL, EDGES_GLOBAL, SCORES_BEFORE_GLOBAL
    G_GLOBAL = G
    EDGES_GLOBAL = edges
    SCORES_BEFORE_GLOBAL = scores_before

In [8]:
def run_dynamic_round(edges,
                      scores,
                      out_edges,
                      in_edges,
                      edges_dict,
                      num_procs,
                      backward_weight,     # tracked BW (sum of weights of backward edges)
                      improvement_counter,
                      index_to_node,
                      output_excel,
                      log_path,
                      BW_SANITY_EVERY_IMPROVEMENTS=5,
                      edge_subset=None):
    """
    Runs one dynamic round:
      - Build current backward edges (u,v,w) where rank(u) > rank(v)
      - Repeatedly select a non-overlapping subset via DP
      - Dispatch each subset to workers to propose local reorders
      - Apply successful changes and update *backward_weight* by subtracting BW reductions

    Logging:
      - Only uses log_message(..., log_path)
      - No other prints/loggers

    Sanity:
      - Periodically recompute BW as: total_edge_weight - forward_weight(scores)
        (since total weight is constant and ranks are injective)
    """
    import math
    import time
    import multiprocessing as mp

    # ----------------- WARN THRESHOLDS -----------------
    BUILD_BACKWARD_WARN = 10.0
    SPAWN_WORKERS_WARN = 5.0
    RESULT_LOOP_WARN = 60.0
    JOIN_WARN = 5.0
    ROUND_TIME_WARN = 600.0
    AVG_WORKER_EDGE_WARN = 0.20
    LOW_SUCCESS_RATIO_WARN = 0.02
    BW_MISMATCH_TOL = 1e-6

    # Total weight is constant → BW = total - FW
    total_edge_weight = 0.0
    for (_, _, w) in edges:
        total_edge_weight += float(w)

    # ----------------- HELPER: SAVE RANKING -----------------
    def save_ranking_snapshot_local(scores_dict, idx_to_node, path):
        # Lazy import to avoid paying import cost if never called
        import pandas as pd

        items = sorted(scores_dict.items(), key=lambda kv: kv[1])  # sort by rank
        rows = []
        for node_idx, rank in items:
            rows.append({"Node ID": str(idx_to_node[node_idx]), "Order": int(rank)})

        df = pd.DataFrame(rows)
        lower = path.lower()
        if lower.endswith(".csv"):
            df.to_csv(path, index=False)
        else:
            df.to_excel(path, index=False)

    round_start = time.time()
    BW_before_round = float(backward_weight)

    # ----------------- TIMING ACCUMULATORS -----------------
    t_build_backward = 0.0
    t_spawn_workers_total = 0.0
    t_result_loop_total = 0.0
    t_join_total = 0.0
    t_dp_total = 0.0

    # Worker timing aggregation
    total_apply_strategy_time_workers = 0.0
    total_greedy_time_workers = 0.0
    total_between_time_workers = 0.0
    total_edge_time_workers = 0.0
    counted_timing_results_total = 0

    total_delta_round = 0.0  # total BW reduction in this round (positive)
    applied_edges_total = 0
    results_processed_total = 0

    # ----- Normalize edge_subset → subset_pairs of (u,v) -----
    subset_pairs = None
    if edge_subset is not None:
        norm_set = set()
        for e in edge_subset:
            if isinstance(e, tuple) and len(e) >= 2:
                norm_set.add((e[0], e[1]))
        subset_pairs = norm_set

    # ----- Build backward edges for this sweep -----
    t0 = time.time()

    backward_edges = []  # list of (u, v, w, lo, hi)
    skipped_by_subset = 0
    skipped_not_backward = 0

    for (u, v, w) in edges:
        if subset_pairs is not None and (u, v) not in subset_pairs:
            skipped_by_subset += 1
            continue

        ru, rv = scores[u], scores[v]
        if ru > rv:
            lo = rv
            hi = ru
            backward_edges.append((u, v, float(w), lo, hi))
        else:
            skipped_not_backward += 1

    backward_edges.sort(key=lambda x: x[2], reverse=True)
    n_edges = len(backward_edges)
    t_build_backward = time.time() - t0

    if t_build_backward > BUILD_BACKWARD_WARN:
        log_message(
            f"WARNING: Building backward edges took {t_build_backward:.4f}s (>{BUILD_BACKWARD_WARN}s)",
            log_path
        )

    if n_edges == 0:
        total_round_time = time.time() - round_start
        log_message(f"No backward edges; ending dynamic round. total_round_time={total_round_time:.4f}s", log_path)
        log_message(
            f"BW_after_round={backward_weight:.2f} (ΔBW_round={BW_before_round - backward_weight:.3f})",
            log_path
        )
        return scores, backward_weight, 0.0, 0, improvement_counter

    # -------- shared structures (kept for signature compatibility) --------
    manager = mp.Manager()
    active_intervals = manager.list()  # unused now
    used_intervals = manager.list()    # unused now
    lock = manager.Lock()              # unused now

    scores_snapshot = scores.copy()

    # Iteratively pick independent subsets via DP
    remaining_indices = list(range(n_edges))
    batch_idx = 0

    while remaining_indices:
        batch_idx += 1
        batch_start = time.time()

        edges_in_pool = len(remaining_indices)
       # log_message(f"DP batch {batch_idx}: edges_in_pool_before_batch={edges_in_pool}", log_path)

        # --- Build local list of edges for DP ---
        edges_for_dp = [backward_edges[i] for i in remaining_indices]

        # --- DP to select a maximum non-conflicting subset ---
        t_dp0 = time.time()
        local_selected = select_nonconflicting_edge_indices_dp(edges_for_dp)
        dp_time = time.time() - t_dp0
        t_dp_total += dp_time

        selected_global = [remaining_indices[i] for i in local_selected]
        num_selected = len(selected_global)

        # log_message(
        #     f"(DP batch {batch_idx}) DP selection: total_edges={len(edges_for_dp)}, selected={num_selected}",
        #     log_path
        # )

        if num_selected == 0:
            log_message(
                f"(DP batch {batch_idx}) DP selected none; stopping with remaining={len(remaining_indices)}",
                log_path
            )
            break

        # --- Build batch_edges list for workers ---
        batch_edges = [backward_edges[i] for i in selected_global]
        num_edges_batch = len(batch_edges)
        num_procs_effective = min(num_procs, num_edges_batch)

        # log_message(
        #     f"(DP batch {batch_idx}) Spawning {num_procs_effective} workers for {num_edges_batch} edges.",
        #     log_path
        # )

        edge_queue = manager.Queue()
        result_queue = manager.Queue()

        for i in range(num_edges_batch):
            edge_queue.put(i)

        # --- Spawn workers ---
        t_spawn0 = time.time()
        procs = []
        for worker_idx in range(1, num_procs_effective + 1):
            p = mp.Process(
                target=worker_loop,
                args=(
                    worker_idx,
                    num_procs_effective,
                    batch_edges,
                    scores_snapshot,
                    out_edges,
                    in_edges,
                    edges_dict,
                    active_intervals,
                    used_intervals,
                    lock,
                    edge_queue,
                    result_queue,
                    log_path
                )
            )
            p.start()
            procs.append(p)

        t_spawn_workers = time.time() - t_spawn0
        t_spawn_workers_total += t_spawn_workers
        if t_spawn_workers > SPAWN_WORKERS_WARN:
            log_message(
                f"WARNING: Spawning workers batch {batch_idx} took {t_spawn_workers:.4f}s (>{SPAWN_WORKERS_WARN}s)",
                log_path
            )

        # --- Collect results ---
        t_res0 = time.time()
        alive = num_procs_effective

        total_delta_batch = 0.0  # BW reduction in this batch
        applied_edges_batch = 0
        results_processed_batch = 0

        # Worker timing aggregation for this batch
        batch_apply_time = 0.0
        batch_greedy_time = 0.0
        batch_between_time = 0.0
        batch_edge_total_time = 0.0
        counted_timing_results_batch = 0

        last_progress_log = time.time()
        BW_before_batch = float(backward_weight)

        while alive > 0:
            res = result_queue.get()

            if res.get("done"):
                alive -= 1
                continue

            results_processed_batch += 1
            results_processed_total += 1

            timing = res.get("timing")
            if timing:
                ta = float(timing.get("apply_new_strategy", 0.0))
                tg = float(timing.get("greedy_time", 0.0))
                tb = float(timing.get("between_list_time", 0.0))
                te = float(timing.get("edge_total_time", 0.0))

                batch_apply_time += ta
                batch_greedy_time += tg
                batch_between_time += tb
                batch_edge_total_time += te
                counted_timing_results_batch += 1

                total_apply_strategy_time_workers += ta
                total_greedy_time_workers += tg
                total_between_time_workers += tb
                total_edge_time_workers += te
                counted_timing_results_total += 1

            if not res.get("success"):
                if time.time() - last_progress_log > 5.0:
                    last_progress_log = time.time()
                continue

            # Success: apply changes
            delta = float(res["delta"])  # POSITIVE BW reduction
            changed = res["changed_scores"]
            u = res["u"]
            v = res["v"]
            mode = res.get("mode", "unknown")

            for node, nrank in changed.items():
                scores[node] = nrank

            backward_weight -= delta
            total_delta_batch += delta
            total_delta_round += delta
            applied_edges_batch += 1
            applied_edges_total += 1
            improvement_counter += 1

            log_message(
                f"(DP batch {batch_idx}) SUCCESS ({u}->{v}) BW_Reduct={delta:.3f} mode={mode} BW={backward_weight:.2f}",
                log_path
            )

            # Periodic sanity check + checkpoint save
            if BW_SANITY_EVERY_IMPROVEMENTS > 0 and (improvement_counter % BW_SANITY_EVERY_IMPROVEMENTS == 0):
                # Compute forward weight, then derive backward weight from total
                fw = 0.0
                for (u2, v2, w2) in edges:
                    if scores[u2] < scores[v2]:
                        fw += float(w2)
                real_bw = total_edge_weight - fw

                if abs(real_bw - backward_weight) > BW_MISMATCH_TOL:
                    log_message(
                        f"RUNTIME ERROR: BW mismatch! tracked={backward_weight:.6f}, recomputed={real_bw:.6f}",
                        log_path
                    )
                    raise RuntimeError("Backward-weight mismatch detected before saving ranking.")

                backward_weight = real_bw  # sync

                try:
                    save_ranking_snapshot_local(scores, index_to_node, output_excel)
                except Exception as e:
                    log_message(
                        f"WARNING: Failed to save intermediate ranking to {output_excel}: {repr(e)}",
                        log_path
                    )

            if time.time() - last_progress_log > 5.0:
                last_progress_log = time.time()

        t_result_loop = time.time() - t_res0
        t_result_loop_total += t_result_loop
        if t_result_loop > RESULT_LOOP_WARN:
            log_message(
                f"WARNING: Result collection loop batch {batch_idx} took {t_result_loop:.4f}s (>{RESULT_LOOP_WARN}s)",
                log_path
            )

        # Join workers
        t_join0 = time.time()
        for p in procs:
            p.join()
        t_join = time.time() - t_join0
        t_join_total += t_join
        if t_join > JOIN_WARN:
            log_message(
                f"WARNING: Joining workers batch {batch_idx} took {t_join:.4f}s (>{JOIN_WARN}s)",
                log_path
            )

        # Remove selected edges from remaining pool
        selected_global_set = set(selected_global)
        remaining_indices = [i for i in remaining_indices if i not in selected_global_set]

        # Batch summary
        batch_total_time = time.time() - batch_start
        log_message(f"(DP batch {batch_idx}) BATCH SUMMARY:", log_path)
        log_message(f"    edges={num_edges_batch}, applied={applied_edges_batch}, BW_Reduct_total={total_delta_batch:.3f}", log_path)
        log_message(f"    BW_before={BW_before_batch:.2f}, BW_after={backward_weight:.2f}", log_path)
        log_message(f"    batch_total_time={batch_total_time:.4f}s", log_path)

    # --------------- DYNAMIC ROUND SUMMARY ---------------
    total_round_time = time.time() - round_start
    avg_gain = (total_delta_round / applied_edges_total) if applied_edges_total > 0 else 0.0
    edges_per_sec = (applied_edges_total / total_round_time) if total_round_time > 0 else 0.0
    avg_worker_edge_time = (
        total_edge_time_workers / counted_timing_results_total
        if counted_timing_results_total > 0 else float('nan')
    )

    log_message("DYNAMIC ROUND SUMMARY:", log_path)
    log_message(
        f"    BW_before={BW_before_round:.2f}, BW_after={backward_weight:.2f}, BW_Reduct_round={total_delta_round:.3f}",
        log_path
    )
    log_message(f"    applied_edges={applied_edges_total}, avg_bw_reduct_per_edge={avg_gain:.4f}", log_path)
    log_message(f"    total_round_time={total_round_time:.4f}s, edges_per_sec={edges_per_sec:.2f}", log_path)
    log_message(
        f"    worker_timing_total: apply≈{total_apply_strategy_time_workers:.4f}s, greedy≈{total_greedy_time_workers:.4f}s",
        log_path
    )

    # --------------- WARNINGS ---------------
    if total_round_time > ROUND_TIME_WARN:
        log_message(f"WARNING: Dynamic round took {total_round_time:.4f}s (>{ROUND_TIME_WARN}s)", log_path)

    if not math.isnan(avg_worker_edge_time) and avg_worker_edge_time > AVG_WORKER_EDGE_WARN:
        log_message(f"WARNING: avg_worker_edge_time={avg_worker_edge_time:.4f}s (>{AVG_WORKER_EDGE_WARN}s)", log_path)

    if results_processed_total > 0:
        success_ratio = applied_edges_total / results_processed_total
        if success_ratio < LOW_SUCCESS_RATIO_WARN:
            log_message(
                f"WARNING: Very low success ratio: {success_ratio:.4%} ({applied_edges_total}/{results_processed_total})",
                log_path
            )

    log_message("END dynamic round (DP-based independent subsets)", log_path)

    return scores, backward_weight, total_delta_round, applied_edges_total, improvement_counter


In [9]:
import csv

def save_scores_to_csv(scores, output_path, index_to_node):
    """
    Saves scores to CSV in the exact format required by load_initial_scores():
    columns = ['Node ID', 'Order'].
    Values:
        Node ID = original node label (string)
        Order   = rank (int)
    Rows sorted by Order.
    """

    # Build list of (order, node_index)
    rows = [(order, node_idx) for node_idx, order in scores.items()]
    rows.sort(key=lambda x: x[0])   # sort by Order

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Node ID", "Order"])  # required header

        for order, node_idx in rows:
            node_label = index_to_node[node_idx]  # original ID string
            writer.writerow([node_label, int(order)])


In [10]:
def refine_ranking_parallel_dynamic(
    csv_path,
    initial_ranking_path,
    output_excel,
    log_path,
    MAX_HOURS: float = 72.0,
    LOG_EVERY_ROUND: int = 1,
    BW_CHECK_EVERY_ROUND: int = 5,
    edge_subset=None,   # None => global; otherwise list of edges; we normalize to (u,v)
):
    """
    Dynamic parallel refinement over a fixed candidate edge pool.

    Tracks and reports only Backward Weight (BW):
      BW(scores) = sum_{(u,v,w) in edges, rank(u) > rank(v)} w

    Key behavior (unchanged structurally):
      - Build a fixed candidate list of edges (possibly a subset).
      - Maintain remaining_indices over this list.
      - Each DP round:
          * Take a fresh snapshot of scores.
          * Among remaining edges, keep those that are backward in snapshot and build (u,v,w,lo,hi).
          * Use DP to choose a maximum non-conflicting subset by [lo,hi].
          * Give only that subset to workers with THIS snapshot.
          * Collect results, apply changed_scores globally (sync point).
          * Remove those edges from remaining_indices (they're assessed).
      - Stop when:
          * no remaining backward edges, OR
          * DP selects zero edges, OR
          * global time limit (MAX_HOURS), OR
          * NO IMPROVEMENT for 10 minutes (wall clock).

    RETURN:
      best_scores, best_bw, remaining_pairs

      where:
        - best_scores: dict node -> rank of the best (lowest BW) seen
        - best_bw: backward weight of best_scores
        - remaining_pairs: set of (u, v) for candidate edges never assessed
    """
    import time
    import datetime
    import os
    import multiprocessing as mp

    pid_main = os.getpid()

    try:
        num_procs = len(os.sched_getaffinity(0))
    except Exception:
        num_procs = os.cpu_count() or 1

    # ---------------------------------------------------
    # Helper: DP independence checker (logs via log_message)
    # ---------------------------------------------------
    def check_dp_interval_independence(edges_for_dp, local_selected, round_idx):
        """
        edges_for_dp: list of (u,v,w,lo,hi) for remaining edges (for THIS snapshot)
        local_selected: indices into edges_for_dp returned by DP
        Ensures selected intervals [lo,hi] are strictly non-overlapping.
        """
        if not local_selected:
            log_message(f"[MAIN] (Round {round_idx}) DP independence check: no edges selected.", log_path)
            return

        tmp = []
        for li in local_selected:
            u, v, w, lo, hi = edges_for_dp[li]
            tmp.append((lo, hi, li, u, v))

        tmp.sort(key=lambda x: x[0])  # sort by lo

        last_lo, last_hi, last_li, last_u, last_v = tmp[0]
        # Strictly disjoint: hi_prev < lo_curr
        for i in range(1, len(tmp)):
            lo, hi, li, u, v = tmp[i]
            if lo <= last_hi:
                log_message(
                    f"[MAIN] DP INDEPENDENCE VIOLATION in round {round_idx}: intervals overlap. "
                    f"A(local_idx={last_li}, {last_u}->{last_v}, [{last_lo},{last_hi}]) "
                    f"B(local_idx={li}, {u}->{v}, [{lo},{hi}])",
                    log_path
                )
                raise RuntimeError("DP selected overlapping intervals; independence broken.")
            if hi > last_hi:
                last_lo, last_hi, last_li, last_u, last_v = lo, hi, li, u, v

        log_message(
            f"[MAIN] (Round {round_idx}) DP independence check PASSED: {len(local_selected)} intervals, no overlaps.",
            log_path
        )

    # ---------------------------------------------------
    # Start logs
    # ---------------------------------------------------
    log_message("Starting DYNAMIC parallel refinement (multi-DP rounds over fixed edge pool)", log_path)
    log_message(f"[MAIN] Graph: {csv_path}", log_path)
    log_message(f"[MAIN] Initial ranking: {initial_ranking_path}", log_path)
    log_message(f"[MAIN] Output ranking: {output_excel}", log_path)
    log_message(f"[MAIN] Available processors = {num_procs}", log_path)

    # ---------------------------------------------------
    # Load graph & initial scores
    # ---------------------------------------------------
    log_message("[MAIN] Reading graph...", log_path)
    edges, node_to_index, index_to_node = read_graph(csv_path)
    log_message(f"[MAIN] Graph loaded: n_nodes={len(node_to_index)}, n_edges={len(edges)}", log_path)

    log_message("[MAIN] Loading initial scores/ranking...", log_path)
    scores = load_initial_scores(initial_ranking_path, node_to_index)
    log_message(f"[MAIN] Loaded scores for {len(scores)} nodes.", log_path)

    # Total edge weight (constant)
    total_weight = 0.0
    for (_, _, w) in edges:
        total_weight += float(w)

    # Compute BW from FW via constant total (your requested trick)
    fw0 = compute_forward_weight(edges, scores)
    bw0 = total_weight - float(fw0)

    ratio_bw = (bw0 / total_weight) if total_weight > 0 else 0.0
    log_message(f"[MAIN] Initial BW / total = {bw0:.2f} / {total_weight:.2f} = {ratio_bw:.6f}", log_path)

    # ---------------------------------------------------
    # Build adjacency & edges_dict
    # ---------------------------------------------------
    log_message("[MAIN] Building adjacency lists and edges_dict...", log_path)
    out_edges = {}
    in_edges = {}
    edges_dict = {}
    for (u, v, w) in edges:
        out_edges.setdefault(u, []).append((v, float(w)))
        in_edges.setdefault(v, []).append((u, float(w)))
        edges_dict[(u, v)] = float(w)

    log_message(
        f"[MAIN] Adjacency built: len(out_edges)={len(out_edges)}, "
        f"len(in_edges)={len(in_edges)}, len(edges_dict)={len(edges_dict)}",
        log_path
    )

    # ---------------------------------------------------
    # Normalize edge_subset → subset_pairs of form (u,v)
    # ---------------------------------------------------
    subset_pairs = None
    if edge_subset is not None:
        raw_len = len(edge_subset)
        norm_set = set()
        malformed = 0

        for e in edge_subset:
            if isinstance(e, tuple) and len(e) >= 2:
                norm_set.add((e[0], e[1]))
            else:
                malformed += 1

        subset_pairs = norm_set
        log_message(
            f"[MAIN] Subset provided: raw_len={raw_len}, normalized_len={len(subset_pairs)}, malformed_entries={malformed}",
            log_path
        )

        # Sample existence check
        sample_pairs = list(subset_pairs)[:10]
        for (su, sv) in sample_pairs:
            exists = (su, sv) in edges_dict
            log_message(f"[MAIN] subset sample edge ({su}->{sv}) exists_in_graph={exists}", log_path)

    # ---------------------------------------------------
    # Fixed candidate pool for whole run
    # ---------------------------------------------------
    candidate_edges = []
    for (u, v, w) in edges:
        if subset_pairs is not None and (u, v) not in subset_pairs:
            continue
        candidate_edges.append((u, v, float(w)))

    log_message(f"[MAIN] Candidate edge pool size = {len(candidate_edges)} (after subset filter).", log_path)

    if subset_pairs is not None and len(candidate_edges) == 0:
        log_message(
            "[MAIN] ERROR: edge_subset (after normalization) is non-empty but no graph edges matched it. "
            "Likely node-ID vs index mismatch or wrong edge direction.",
            log_path
        )
        raise RuntimeError("edge_subset does not match any edges in this graph.")

    remaining_indices = list(range(len(candidate_edges)))

    # Best = minimum BW
    best_scores = scores.copy()
    best_BW = float(bw0)

    # Tracked current BW
    BW_tracked = float(bw0)

    log_message(f"[MAIN] Initial best BW set to {best_BW:.2f}", log_path)

    start_ts = time.time()
    deadline = start_ts + MAX_HOURS * 3600.0

    NO_IMPROVEMENT_TIME_LIMIT_SEC = 600.0
    last_improvement_time = start_ts

    BW_MISMATCH_TOL = 1e-6
    round_idx = 0

    # ===================================================
    # MAIN LOOP: one DP round per iteration
    # ===================================================
    while remaining_indices:
        now = time.time()
        if now >= deadline:
            log_message("[MAIN] Time limit reached before starting new round; stopping.", log_path)
            break

        round_idx += 1
        elapsed_h = (now - start_ts) / 3600.0
        time_left_h = max(0.0, (deadline - now) / 3600.0)
        since_improvement = now - last_improvement_time

        if LOG_EVERY_ROUND > 0 and (round_idx % LOG_EVERY_ROUND == 0):
            log_message("", log_path)
            log_message(f"[MAIN] ===== Dynamic DP round {round_idx} =====", log_path)
            log_message(
                f"[MAIN]    Current BW(tracked)={BW_tracked:.2f}, best_BW={best_BW:.2f}, "
                f"elapsed={elapsed_h:.2f}h, time_left={time_left_h:.2f}h",
                log_path
            )
            log_message(f"[MAIN]    Remaining candidate edges: {len(remaining_indices)}", log_path)
            log_message(
                f"[MAIN]    Time since last improvement: {since_improvement:.1f}s "
                f"(limit={NO_IMPROVEMENT_TIME_LIMIT_SEC:.0f}s)",
                log_path
            )

        # Fresh snapshot for THIS round
        scores_snapshot = scores.copy()

        # ---------------------------------------------------
        # Build edges_for_dp among remaining, using snapshot
        # ---------------------------------------------------
        edges_for_dp = []
        global_idx_for_local = []
        num_non_backward = 0

        for gi in remaining_indices:
            u, v, w = candidate_edges[gi]
            ru = scores_snapshot[u]
            rv = scores_snapshot[v]
            if ru > rv:
                lo = rv
                hi = ru
                edges_for_dp.append((u, v, w, lo, hi))
                global_idx_for_local.append(gi)
            else:
                num_non_backward += 1

        if LOG_EVERY_ROUND > 0 and (round_idx % LOG_EVERY_ROUND == 0):
            log_message(
                f"[MAIN] (Round {round_idx}) remaining={len(remaining_indices)}, "
                f"backward_in_snapshot={len(edges_for_dp)}, non_backward={num_non_backward}",
                log_path
            )

        if not edges_for_dp:
            log_message(f"[MAIN] (Round {round_idx}) No backward edges among remaining candidates; stopping.", log_path)
            break

        # ---------------------------------------------------
        # DP selection
        # ---------------------------------------------------
        local_selected = select_nonconflicting_edge_indices_dp(edges_for_dp)
        if not local_selected:
            log_message(f"[MAIN] (Round {round_idx}) DP selected zero edges; stopping.", log_path)
            break

        check_dp_interval_independence(edges_for_dp, local_selected, round_idx)

        selected_global_indices = [global_idx_for_local[li] for li in local_selected]
        batch_edges = [edges_for_dp[li] for li in local_selected]  # (u,v,w,lo,hi)
        num_edges_batch = len(batch_edges)

        # ------------------ Spawn workers for this batch ------------------
        num_procs_effective = min(num_procs, num_edges_batch)

        edge_queue = mp.Queue()
        result_queue = mp.Queue()
        for i in range(num_edges_batch):
            edge_queue.put(i)

        dummy_manager = mp.Manager()
        active_intervals = dummy_manager.list()
        used_intervals = dummy_manager.list()
        lock = dummy_manager.Lock()

        workers = []
        for wi in range(num_procs_effective):
            p = mp.Process(
                target=worker_loop,
                args=(
                    wi + 1,
                    num_procs_effective,
                    batch_edges,
                    scores_snapshot,
                    out_edges,
                    in_edges,
                    edges_dict,
                    active_intervals,
                    used_intervals,
                    lock,
                    edge_queue,
                    result_queue,
                    log_path,            # IMPORTANT: pass log_path
                )
            )
            p.daemon = False
            p.start()
            workers.append(p)

        # ------------------ Collect results ------------------
        alive_workers = num_procs_effective
        results_processed = 0

        used_nodes_this_batch = set()
        total_bw_reduct_round = 0.0
        applied_edges_round = 0

        while alive_workers > 0:
            res = result_queue.get()

            if res.get("done"):
                alive_workers -= 1
                continue

            results_processed += 1

            success = bool(res.get("success"))
            delta = float(res.get("delta", 0.0))  # BW reduction
            changed_scores = res.get("changed_scores", {})
            u = res.get("u")
            v = res.get("v")
            mode = res.get("mode", "none")

            if success and delta > 0.0:
                changed_nodes = set(changed_scores.keys())
                intersect = used_nodes_this_batch.intersection(changed_nodes)
                if intersect:
                    log_message(
                        f"[MAIN] NODE-LEVEL INDEPENDENCE VIOLATION in round {round_idx}: "
                        f"nodes changed by multiple edges in same DP batch. example={list(intersect)[:10]}",
                        log_path
                    )
                    raise RuntimeError("DP independence violated at node level (changed_scores).")

                used_nodes_this_batch.update(changed_nodes)

                applied_edges_round += 1
                total_bw_reduct_round += delta
                BW_tracked -= delta  # BW decreases when improved

                for node, new_r in changed_scores.items():
                    scores[node] = new_r

                last_improvement_time = time.time()

                log_message(
                    f"[MAIN] (Round {round_idx}) SUCCESS ({u}->{v}) BW_Reduct={delta:.3f} mode={mode} changed_nodes={len(changed_scores)}",
                    log_path
                )

        # Join workers
        for p in workers:
            p.join()

        # Remove selected edges from remaining_indices
        selected_global_set = set(selected_global_indices)
        remaining_indices = [idx for idx in remaining_indices if idx not in selected_global_set]

        # Round summary
        if LOG_EVERY_ROUND > 0 and (round_idx % LOG_EVERY_ROUND == 0):
            log_message(
                f"[MAIN] Round {round_idx} summary: BW_Reduct_round={total_bw_reduct_round:.3f}, applied_edges_round={applied_edges_round}",
                log_path
            )
            log_message(
                f"[MAIN] Round {round_idx} end: BW_tracked={BW_tracked:.2f}, best_BW={best_BW:.2f}, remaining_candidates={len(remaining_indices)}",
                log_path
            )

        # Update global best if BW improved (smaller is better)
        improved_this_round = False
        if BW_tracked < best_BW:
            best_BW = BW_tracked
            best_scores = scores.copy()
            improved_this_round = True
            log_message(f"[MAIN] New best BW={best_BW:.2f} (updated best_scores after round {round_idx})", log_path)

        # Save intermediate ranking ONLY if improved global best
        if improved_this_round:
            # Prefer the same writer you used elsewhere; here we reuse save_scores_to_csv if you already have it.
            # If your save_scores_to_csv writes Excel, that's fine; just keep it consistent in your codebase.
            save_scores_to_csv(best_scores, output_excel, index_to_node)
            log_message(f"[MAIN] Saved intermediate BEST ranking (round {round_idx}, BW={best_BW:.2f}) to {output_excel}", log_path)

        # BW sanity check every BW_CHECK_EVERY_ROUND rounds
        if BW_CHECK_EVERY_ROUND > 0 and (round_idx % BW_CHECK_EVERY_ROUND == 1):
            fw_recompute = compute_forward_weight(edges, scores)
            bw_recompute = total_weight - float(fw_recompute)
            if abs(bw_recompute - BW_tracked) > BW_MISMATCH_TOL:
                log_message(
                    f"[MAIN] SANITY MISMATCH after round {round_idx}: "
                    f"BW_recomputed={bw_recompute:.6f}, BW_tracked={BW_tracked:.6f}",
                    log_path
                )
                raise RuntimeError("Backward-weight tracking mismatch.")
            else:
                log_message(
                    f"[MAIN] SANITY after round {round_idx}: BW_recomputed={bw_recompute:.6f}, BW_tracked={BW_tracked:.6f}",
                    log_path
                )

        # No-improvement stop
        no_improvement_elapsed = time.time() - last_improvement_time
        if no_improvement_elapsed >= NO_IMPROVEMENT_TIME_LIMIT_SEC:
            log_message(
                f"[MAIN] Stopping after round {round_idx}: no improvement for {no_improvement_elapsed:.1f}s "
                f"(limit={NO_IMPROVEMENT_TIME_LIMIT_SEC:.0f}s).",
                log_path
            )
            break

        if not remaining_indices:
            log_message("[MAIN] All candidate edges assessed; stopping.", log_path)
            break

    # ---------------------------------------------------
    # Final write of best_scores
    # ---------------------------------------------------
    save_scores_to_csv(best_scores, output_excel, index_to_node)
    final_fw = compute_forward_weight(edges, best_scores)
    final_bw = total_weight - float(final_fw)

    log_message(f"[MAIN] Finished. Final best BW={final_bw:.2f}. Ranking written to {output_excel}", log_path)

    remaining_pairs = set()
    for gi in remaining_indices:
        u, v, w = candidate_edges[gi]
        remaining_pairs.add((u, v))
    log_message(f"[MAIN] Unassessed edges remaining: {len(remaining_pairs)}", log_path)

    return best_scores, final_bw, remaining_pairs


In [11]:
# ============================================
# Block 5: func2 – SCC block refinement (largest SCC intervals)
# ============================================

def _compute_block_fw(G, scores, block_nodes):
    subG = G.subgraph(block_nodes)
    fw = 0.0
    for u, v, data in subG.edges(data=True):
        w = data.get('weight', 1.0)
        if scores[u] < scores[v]:
            fw += w
    return fw


def _worker_refine_block(args):
    (
        worker_id,
        block_nodes,
        brute_force_min_size,
        brute_force_max_size,
        debug
    ) = args

    import os, time
    from datetime import datetime

    # Use global objects initialized via _init_refine_worker
    global G_GLOBAL, EDGES_GLOBAL, SCORES_BEFORE_GLOBAL
    G = G_GLOBAL
    edges = EDGES_GLOBAL
    scores_before = SCORES_BEFORE_GLOBAL

    pid = os.getpid()
    block_nodes = list(block_nodes)

    # --- Start timestamp ---
    t_start = time.time()
    start_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if debug:
        print(f"[{worker_id} | PID {pid} | START {start_ts}] 🧵 starting on block with {len(block_nodes)} nodes")

    fw_block_before = _compute_block_fw(G, scores_before, block_nodes)
    if debug:
        print(f"[{worker_id} | PID {pid}] block FW BEFORE = {fw_block_before:.6f}")

    # Call refine_block_scc_interval
    new_scores, _, _, _ = refine_block_scc_interval(
        G=G,
        edges=edges,
        node_order=scores_before,
        block_nodes=block_nodes,
        brute_force_min_size=brute_force_min_size,
        brute_force_max_size=brute_force_max_size,
        debug=False
    )

    fw_block_after = _compute_block_fw(G, new_scores, block_nodes)
    if debug:
        print(f"[{worker_id} | PID {pid}] block FW AFTER  = {fw_block_after:.6f}")

    if fw_block_after + 1e-9 < fw_block_before:
        raise RuntimeError(
            f"[{worker_id} | PID {pid}] ❌ block FW decreased: "
            f"before={fw_block_before:.6f}, after={fw_block_after:.6f}"
        )

    block_ranks = {n: new_scores[n] for n in block_nodes}

    # --- End timestamp ---
    t_end = time.time()
    end_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    elapsed = t_end - t_start

    if debug:
        print(
            f"[{worker_id} | PID {pid} | END {end_ts}] "
            f"✅ finished, ΔFW_block = {fw_block_after - fw_block_before:.6f}, "
            f"elapsed = {elapsed:.2f} s"
        )

    return {
        "worker_id": worker_id,
        "pid": pid,
        "block_nodes": block_nodes,
        "block_ranks": block_ranks,
        "fw_block_before": fw_block_before,
        "fw_block_after": fw_block_after,
        "elapsed_worker": elapsed,
    }


def refine_block_scc_interval(
    G,
    edges,
    node_order,
    block_nodes,
    brute_force_min_size=2,
    brute_force_max_size=7,
    debug=False,
):
    """
    Refine a given interval (block) of nodes using SCC decomposition + topo sort.

    Returns
    -------
    new_node_order : dict[node -> int]
        Updated global ranking after refining this block.
    existing_fw : float
        Total forward weight before refinement (global).
    new_fw : float
        Total forward weight after refinement (global).
    improved : bool
        True iff new_fw > existing_fw.
    """
    import networkx as nx
    from itertools import permutations
    from collections import Counter

    block_nodes = list(block_nodes)
    if debug:
        print(f"\n=== refine_block_scc_interval on block of size {len(block_nodes)} ===")

    # Sort block nodes by current rank for stability
    block_nodes = sorted(block_nodes, key=lambda n: node_order[n])

    # Copy ranking
    prev_order = node_order.copy()
    new_node_order = node_order.copy()

    # Subgraph induced by the block
    subG = G.subgraph(block_nodes).copy()
    if debug:
        print(f"   subgraph: |V|={subG.number_of_nodes()}, |E|={subG.number_of_edges()}")

    # SCC decomposition inside block
    sub_sccs = list(nx.strongly_connected_components(subG))
    if debug:
        print(f"   found {len(sub_sccs)} SCCs in block")

    # Map node -> SCC id
    node_to_subscc = {}
    for idx, scc in enumerate(sub_sccs):
        for n in scc:
            node_to_subscc[n] = idx

    # Build SCC DAG
    scc_dag = nx.DiGraph()
    scc_dag.add_nodes_from(range(len(sub_sccs)))
    for u, v in subG.edges():
        su = node_to_subscc[u]
        sv = node_to_subscc[v]
        if su != sv:
            scc_dag.add_edge(su, sv)

    # Topological order of SCCs
    scc_order = list(nx.topological_sort(scc_dag))
    if debug:
        print(f"   topo order of SCCs length = {len(scc_order)}")

    # Construct new order for the block
    new_block_order = []
    for scc_id in scc_order:
        scc_nodes = list(sub_sccs[scc_id])
        scc_size = len(scc_nodes)

        if scc_size == 1:
            new_block_order.extend(scc_nodes)

        elif brute_force_min_size <= scc_size <= brute_force_max_size:
            if debug:
                print(f"   SCC {scc_id}: size={scc_size}, brute-forcing permutations")
            best_perm = None
            max_weight = float("-inf")
            for perm in permutations(scc_nodes):
                w_sum = 0.0
                for i, u in enumerate(perm):
                    for j in range(i + 1, scc_size):
                        v = perm[j]
                        if subG.has_edge(u, v):
                            w_sum += subG[u][v]['weight']
                if w_sum > max_weight:
                    max_weight = w_sum
                    best_perm = perm
            if debug:
                print(f"      best internal FW = {max_weight:.2f}")
            new_block_order.extend(best_perm)

        else:
            if debug:
                print(f"   SCC {scc_id}: size={scc_size}, keeping original order")
            sorted_original = sorted(scc_nodes, key=lambda n: prev_order[n])
            new_block_order.extend(sorted_original)

    # Sanity: permutation
    if set(new_block_order) != set(block_nodes) or len(new_block_order) != len(block_nodes):
        if debug:
            missing = set(block_nodes) - set(new_block_order)
            extra = set(new_block_order) - set(block_nodes)
            print(f"❌ mismatch in new_block_order: missing={len(missing)}, extra={len(extra)}")
        raise ValueError("Block nodes mismatch in new_block_order!")

    # Reassign ranks ONLY within the block, reusing original ranks
    orig_ranks = sorted(prev_order[n] for n in block_nodes)
    for i, node in enumerate(new_block_order):
        new_node_order[node] = orig_ranks[i]

    # Global duplicate-rank sanity check
    all_ranks = list(new_node_order.values())
    if len(all_ranks) != len(set(all_ranks)):
        if debug:
            duplicates = [r for r, cnt in Counter(all_ranks).items() if cnt > 1]
            print(f"❌ Duplicate ranks detected — {len(duplicates)} ranks are non-unique")
        raise ValueError("Duplicate ranks detected in new_node_order!")

    # Compute global FW before/after
    existing_fw = compute_forward_weight(edges, prev_order)
    new_fw = compute_forward_weight(edges, new_node_order)
    improved = new_fw > existing_fw

    if debug:
        print(f"   FW before = {existing_fw:.2f}, FW after = {new_fw:.2f}, Δ={new_fw - existing_fw:.2f}")
        if improved:
            print("   ✅ block refinement improved global FW")
        else:
            print("   ⏭️  no global improvement from this block")

    return new_node_order, existing_fw, new_fw, improved


def parallel_refine_largest_scc_intervals(
    block_size=530,                 # kept for API compatibility (unused for SCC range)
    brute_force_min_size=2,
    brute_force_max_size=7,
    max_backward_flips=100,
    verify_every=5,
    csv_path="/content/drive/MyDrive/connectome_graph.csv",
    initial_ranking_path="/content/drive/MyDrive/bader.csv",
    output_path=None,
    save_every=5,                   # save ranking to output_path every 'save_every' batches
    debug=False,                    # ✅ DEFAULT: quiet
    max_no_improvement_batches=10,  # patience (per-SCC) on non-improving batches
    log_path=None,                  # ✅ REQUIRED: no prints, log only
):
    """
    Parallel refinement over ALL non-trivial SCCs (size >= 2) using non-overlapping
    rank-interval blocks.

    CONSISTENCY RULES:
      - NO print() anywhere.
      - Logging ONLY via log_message(msg, log_path).
      - Main metric is Backward Weight (BW). (We track FW internally and convert via BW=total-FW.)
      - Preserves cross-boundary invariant: edges crossing union of all block nodes must not
        change direction within a batch.

    Returns:
      current_scores, bw_history, f2b_edges_global, output_path
        - bw_history: tracked BW after each batch (starts with initial BW)
        - f2b_edges_global: set of edges (u,v) that changed forward->backward in any batch
    """
    import time
    import random
    import os
    import math
    import multiprocessing as mp
    import networkx as nx
    import pandas as pd
    from collections import deque

    if log_path is None:
        raise TypeError("parallel_refine_largest_scc_intervals: log_path is required (cannot be None).")

    # ----------------- logging helpers -----------------
    def _log(msg: str, force: bool = False):
        # force=True => always log even when debug=False
        if force or debug:
            log_message(f"[func2] {msg}", log_path)

    def _log_important(msg: str):
        log_message(f"[func2] {msg}", log_path)

    # ----------------- small local helpers -----------------
    def all_scores_unique(scores_dict):
        vals = list(scores_dict.values())
        return len(vals) == len(set(vals))

    def compute_f2b_b2f_and_delta_fw(edges, sbef, saft):
        """
        Computes:
          sum_f2b: total weight of edges that were forward before and backward after
          sum_b2f: total weight of edges that were backward before and forward after
          f2b_edges: set of (u,v) that flipped forward->backward
        Orientation uses STRICT comparisons:
          forward: rank[u] < rank[v]
          backward: rank[u] > rank[v]
        """
        sum_f2b = 0.0
        sum_b2f = 0.0
        f2b_edges = set()
        for (u, v, w) in edges:
            bu, bv = sbef[u], sbef[v]
            au, av = saft[u], saft[v]

            before_fwd = (bu < bv)
            before_bwd = (bu > bv)
            after_fwd  = (au < av)
            after_bwd  = (au > av)

            if before_fwd and after_bwd:
                sum_f2b += float(w)
                f2b_edges.add((u, v))
            elif before_bwd and after_fwd:
                sum_b2f += float(w)
        delta_fw = sum_b2f - sum_f2b
        return sum_f2b, sum_b2f, delta_fw, f2b_edges

    def _interval_overlaps(a_low, a_high, b_low, b_high):
        return not (a_high < b_low or b_high < a_low)

    def _rank_interval_has_backward_edge(edges_in_scc, scores_loc, r_low, r_high):
        # edges_in_scc are internal SCC edges only (u,v,w)
        for (u, v, _w) in edges_in_scc:
            ru = scores_loc[u]
            rv = scores_loc[v]
            if (r_low <= ru <= r_high) and (r_low <= rv <= r_high) and (ru > rv):
                return True
        return False

    # ----------------- bandit-ish parameters (kept) -----------------
    P_EXPLOIT = 0.4
    P_UCB     = 0.4
    P_RANDOM  = 0.2

    GOOD_INTERVAL_MAX = 50
    COOL_DOWN_BATCHES = 10
    COOL_MIN_DELTA_FW = -1e-9  # only strictly negative deltas mark "cold"

    # ----------------- processors -----------------
    try:
        num_procs = len(os.sched_getaffinity(0))
    except Exception:
        num_procs = mp.cpu_count() or 1

    # ----------------- read graph + scores -----------------
    edges_indexed, node_to_index, index_to_node = read_graph(csv_path)
    scores = load_initial_scores(initial_ranking_path, node_to_index)

    if not all_scores_unique(scores):
        raise RuntimeError("Initial scores are not unique")

    total_weight = 0.0
    for (_u, _v, w) in edges_indexed:
        total_weight += float(w)

    # Build DiGraph once
    G = nx.DiGraph()
    G.add_weighted_edges_from(edges_indexed)

    # output_path
    if output_path is None:
        dir_path, base_name = os.path.split(initial_ranking_path)
        base_root, ext = os.path.splitext(base_name)
        out_name = f"{base_root}_scc_parallel_blocks{ext}"
        output_path = os.path.join(dir_path, out_name)
    else:
        dir_path, base_name = os.path.split(output_path)
        if dir_path == "":
            parent_dir, _ = os.path.split(initial_ranking_path)
            output_path = os.path.join(parent_dir, output_path)

    _log_important(
        f"Start: refine ALL SCCs | procs={num_procs} | max_backward_flips={max_backward_flips} "
        f"| verify_every={verify_every} | save_every={save_every} | output={output_path}"
    )

    # ----------------- global SCC topo reorder (keeps SCCs contiguous) -----------------
    # NOTE: your global_scc_topo_reorder_scores signature is:
    #   global_scc_topo_reorder_scores(G, edges, scores, log_path, debug=False)
    scores = global_scc_topo_reorder_scores(G, edges_indexed, scores, log_path=log_path, debug=debug)

    # ----------------- SCC decomposition (fixed for whole run) -----------------
    sccs = list(nx.strongly_connected_components(G))
    if not sccs:
        raise RuntimeError("No SCCs found in graph (graph empty?).")

    # Keep only SCCs with size >= 2
    scc_list = [set(s) for s in sccs if len(s) >= 2]
    scc_list.sort(key=len, reverse=True)  # big first (good default)

    _log_important(f"SCCs: total={len(sccs)}, nontrivial(size>=2)={len(scc_list)}")

    if not scc_list:
        # nothing to refine
        current_scores = scores.copy()
        fw_current = float(compute_forward_weight(edges_indexed, current_scores))
        bw_current = float(total_weight - fw_current)
        bw_history = [bw_current]

        # final save
        rows = [(index_to_node[idx], current_scores[idx]) for idx in current_scores]
        rows_sorted = sorted(rows, key=lambda x: x[1])
        pd.DataFrame(rows_sorted, columns=["Node ID", "Order"]).to_csv(output_path, index=False)

        _log_important(f"Done: no nontrivial SCCs. Final BW={bw_current:.6f}")
        return current_scores, bw_history, set(), output_path

    # ----------------- precompute SCC internal edges (for fast interval checks) -----------------
    # Map node -> scc_id (in our scc_list indexing)
    node_to_sccid = {}
    for sid, nodeset in enumerate(scc_list):
        for n in nodeset:
            node_to_sccid[n] = sid

    edges_in_scc = [[] for _ in range(len(scc_list))]
    for (u, v, w) in edges_indexed:
        su = node_to_sccid.get(u, None)
        sv = node_to_sccid.get(v, None)
        if su is not None and su == sv:
            edges_in_scc[su].append((u, v, w))

    # ----------------- per-SCC learning state -----------------
    # IMPORTANT: SCC rank span is stable because we only permute ranks within SCC blocks.
    scc_state = {}
    for sid, nodeset in enumerate(scc_list):
        ranks = [scores[n] for n in nodeset]
        mn = min(ranks)
        mx = max(ranks)
        span = mx - mn + 1

        target_buckets = max(num_procs * 4, 8)
        bucket_size = max(1, span // target_buckets)
        num_buckets = (span + bucket_size - 1) // bucket_size

        scc_state[sid] = {
            "active": True,
            "size": len(nodeset),
            "min_rank": mn,
            "max_rank": mx,
            "bucket_size": bucket_size,
            "num_buckets": num_buckets,
            "bucket_fw_gain": [0.0] * num_buckets,
            "bucket_counts":  [0] * num_buckets,
            "total_blocks_sampled": 0,
            "good_intervals": deque(maxlen=GOOD_INTERVAL_MAX),  # (r_low,r_high,delta_fw,batch_idx)
            "cool_intervals": [],                               # (r_low,r_high,batch_idx)
            "no_improve_batches": 0,
        }

    # ----------------- main tracked state -----------------
    current_scores = scores.copy()
    fw_current = float(compute_forward_weight(edges_indexed, current_scores))
    bw_current = float(total_weight - fw_current)
    bw_history = [bw_current]

    _log_important(f"Initial BW={bw_current:.6f} (FW={fw_current:.6f}, totalW={total_weight:.6f})")

    f2b_edges_global = set()
    batch_idx = 0
    t_start = time.time()

    # ================= MAIN BATCH LOOP =================
    while True:
        # Stop: budget
        if len(f2b_edges_global) >= max_backward_flips:
            _log_important(f"Stop: reached max_backward_flips={max_backward_flips} (unique F→B edges).")
            break

        # Stop: all SCCs inactive (stagnant)
        active_sids = [sid for sid, st in scc_state.items() if st["active"]]
        if not active_sids:
            _log_important("Stop: all SCCs reached no-improvement patience (inactive).")
            break

        # -------------- Build ONE batch across SCCs (up to num_procs blocks) --------------
        used_rank_ranges = []  # global: (r_low,r_high)
        batch_blocks = []      # list of dicts: {sid, r_low, r_high, block_nodes}
        max_attempts = num_procs * 50
        attempts = 0

        # helper: UCB pick bucket (SCC-local)
        def _pick_bucket_index(st):
            # only choose among buckets that currently have at least one SCC node
            st["total_blocks_sampled"] += 1
            t = st["total_blocks_sampled"]

            # if never tried: random
            if all(c == 0 for c in st["bucket_counts"]):
                return random.randrange(st["num_buckets"])

            explore_c = 1.0
            best_b = None
            best_score = float("-inf")
            for b in range(st["num_buckets"]):
                n_b = st["bucket_counts"][b]
                if n_b == 0:
                    mean = 0.0
                    bonus = math.sqrt(2.0 * math.log(t + 1.0))
                else:
                    mean = st["bucket_fw_gain"][b] / n_b
                    bonus = explore_c * math.sqrt(math.log(t + 1.0) / n_b)
                score = mean + bonus
                if score > best_score:
                    best_score = score
                    best_b = b
            return best_b if best_b is not None else random.randrange(st["num_buckets"])

        def _in_cooldown(st, r_low, r_high, current_batch_idx):
            for (cl, ch, bidx) in st["cool_intervals"]:
                if current_batch_idx - bidx <= COOL_DOWN_BATCHES:
                    if _interval_overlaps(r_low, r_high, cl, ch):
                        return True
            return False

        while len(batch_blocks) < num_procs and attempts < max_attempts:
            attempts += 1

            # choose an SCC (weighted by size to focus effort where it matters)
            sid = random.choices(
                active_sids,
                weights=[scc_state[x]["size"] for x in active_sids],
                k=1
            )[0]
            st = scc_state[sid]
            nodeset = scc_list[sid]
            n_scc = st["size"]

            # SCC-local min/max block sizes (adapt for small SCCs)
            min_scc_block_size = 2000
            max_scc_block_size = 13000
            if n_scc < min_scc_block_size:
                min_scc_block_size = max(1, n_scc // 2)
                max_scc_block_size = n_scc

            # Current SCC ordering by rank
            scc_nodes_sorted = sorted(list(nodeset), key=lambda n: current_scores[n])

            # Build bucket_to_indices for THIS current ordering
            bucket_to_indices = [[] for _ in range(st["num_buckets"])]
            mn = st["min_rank"]
            bs = st["bucket_size"]
            for idx, node in enumerate(scc_nodes_sorted):
                r = current_scores[node]
                b = (r - mn) // bs
                if 0 <= b < st["num_buckets"]:
                    bucket_to_indices[b].append(idx)

            # decide strategy
            r_strategy = random.random()
            candidate_interval = None  # (r_low, r_high)

            # 1) Exploit around good intervals
            if st["good_intervals"] and r_strategy < P_EXPLOIT:
                base_r_low, base_r_high, _, _ = random.choice(st["good_intervals"])
                width = max(1, base_r_high - base_r_low)

                expand_frac = random.uniform(-0.3, 0.5)
                shift_frac  = random.uniform(-0.3, 0.3)

                new_width = int(max(1, width * (1.0 + expand_frac)))
                r_center = (base_r_low + base_r_high) / 2.0 + shift_frac * width

                r_low  = int(round(r_center - new_width / 2.0))
                r_high = int(round(r_center + new_width / 2.0))
                r_low  = max(r_low, st["min_rank"])
                r_high = min(r_high, st["max_rank"])
                if r_high > r_low:
                    candidate_interval = (r_low, r_high)

            # 2) UCB bucket selection
            elif r_strategy < P_EXPLOIT + P_UCB:
                b = _pick_bucket_index(st)
                idxs = bucket_to_indices[b] if (b is not None and b < len(bucket_to_indices)) else []
                if idxs:
                    center_idx = random.choice(idxs)
                    size_scc = random.randint(min_scc_block_size, max_scc_block_size)
                    size_scc = min(size_scc, n_scc)
                    if size_scc > 0:
                        start_idx = max(0, center_idx - size_scc // 2)
                        end_idx = min(n_scc, start_idx + size_scc)
                        start_idx = max(0, end_idx - size_scc)

                        first_node = scc_nodes_sorted[start_idx]
                        last_node  = scc_nodes_sorted[end_idx - 1]
                        r_low = current_scores[first_node]
                        r_high = current_scores[last_node]
                        if r_high < r_low:
                            r_low, r_high = r_high, r_low
                        candidate_interval = (r_low, r_high)

            # 3) Random interval
            else:
                size_scc = random.randint(min_scc_block_size, max_scc_block_size)
                size_scc = min(size_scc, n_scc)
                if size_scc > 0:
                    start_idx = random.randint(0, n_scc - size_scc)
                    end_idx = start_idx + size_scc
                    first_node = scc_nodes_sorted[start_idx]
                    last_node  = scc_nodes_sorted[end_idx - 1]
                    r_low = current_scores[first_node]
                    r_high = current_scores[last_node]
                    if r_high < r_low:
                        r_low, r_high = r_high, r_low
                    candidate_interval = (r_low, r_high)

            if candidate_interval is None:
                continue

            r_low, r_high = candidate_interval

            # cooldown
            if _in_cooldown(st, r_low, r_high, batch_idx + 1):
                continue

            # global non-overlap check (rank ranges)
            overlap = False
            for (rl, rh) in used_rank_ranges:
                if _interval_overlaps(r_low, r_high, rl, rh):
                    overlap = True
                    break
            if overlap:
                continue

            # block nodes restricted to THIS SCC
            block_nodes = [n for n in nodeset if (r_low <= current_scores[n] <= r_high)]
            if not block_nodes:
                continue

            # must contain a backward edge (inside SCC internal edges only)
            if not _rank_interval_has_backward_edge(edges_in_scc[sid], current_scores, r_low, r_high):
                continue

            batch_blocks.append({"sid": sid, "r_low": r_low, "r_high": r_high, "block_nodes": block_nodes})
            used_rank_ranges.append((r_low, r_high))

        if not batch_blocks:
            # If we can't find any blocks, likely no SCC has backward edges remaining in any interval.
            _log_important("Stop: could not construct any block with a backward edge across active SCCs.")
            break

        # sanity: global ranges strictly non-overlapping
        used_rank_ranges_sorted = sorted(used_rank_ranges)
        for i in range(1, len(used_rank_ranges_sorted)):
            prev_l, prev_h = used_rank_ranges_sorted[i - 1]
            cur_l, cur_h = used_rank_ranges_sorted[i]
            if not (cur_l > prev_h):
                raise RuntimeError(f"Rank-interval overlap detected between {used_rank_ranges_sorted[i-1]} and {used_rank_ranges_sorted[i]}")

        # -------------- Run batch in parallel --------------
        batch_idx += 1
        scores_before_batch = current_scores.copy()
        fw_old = fw_current
        bw_old = float(total_weight - fw_old)

        worker_args = []
        for b_id, blk in enumerate(batch_blocks):
            worker_id = f"batch{batch_idx}_blk{b_id}_scc{blk['sid']}"
            worker_args.append((worker_id, blk["block_nodes"], brute_force_min_size, brute_force_max_size, debug))

        batch_start = time.time()
        with mp.Pool(
            processes=min(num_procs, len(worker_args)),
            initializer=_init_refine_worker,
            initargs=(G, edges_indexed, scores_before_batch)
        ) as pool:
            results = pool.map(_worker_refine_block, worker_args)
        batch_elapsed = time.time() - batch_start

        # merge
        for res in results:
            block_ranks = res["block_ranks"]
            for n, r in block_ranks.items():
                current_scores[n] = r

        if not all_scores_unique(current_scores):
            raise RuntimeError(f"Duplicate scores detected after batch {batch_idx}")

        # cross-boundary invariant
        B = set()
        for blk in batch_blocks:
            B.update(blk["block_nodes"])

        bad_cross = 0
        for (u, v, _w) in edges_indexed:
            in_u = (u in B)
            in_v = (v in B)
            if in_u ^ in_v:
                was_fwd = scores_before_batch[u] < scores_before_batch[v]
                now_fwd = current_scores[u] < current_scores[v]
                if was_fwd != now_fwd:
                    bad_cross += 1
        if bad_cross > 0:
            raise RuntimeError(
                f"{bad_cross} cross-boundary edges changed direction. Block rank-interval invariant broken."
            )

        # update tracked FW/BW using strict direction flips
        sum_f2b, sum_b2f, delta_fw, f2b_edges_batch = compute_f2b_b2f_and_delta_fw(
            edges_indexed, scores_before_batch, current_scores
        )
        fw_current = float(fw_old + delta_fw)
        bw_current = float(total_weight - fw_current)
        bw_history.append(bw_current)

        # update global F→B edge set (budget)
        before_sz = len(f2b_edges_global)
        f2b_edges_global.update(f2b_edges_batch)
        added_sz = len(f2b_edges_global) - before_sz

        # -------- per-SCC learning updates (coarse credit assignment) --------
        # Split delta_fw evenly across blocks as a simple credit heuristic.
        # (Keeps code simple and stable; you can refine later.)
        per_block_reward = (delta_fw / len(batch_blocks)) if batch_blocks else 0.0

        # update SCC-level no-improve streaks only for SCCs used in this batch
        sids_in_batch = set(blk["sid"] for blk in batch_blocks)

        for sid in sids_in_batch:
            st = scc_state[sid]
            # reward signal
            if per_block_reward > 1e-9:
                st["no_improve_batches"] = 0
            else:
                st["no_improve_batches"] += 1
                if (max_no_improvement_batches is not None) and (st["no_improve_batches"] >= max_no_improvement_batches):
                    st["active"] = False

        # bucket updates + good/cool interval memory per block (SCC-local)
        for blk in batch_blocks:
            sid = blk["sid"]
            st = scc_state[sid]

            # buckets intersecting this interval
            mn = st["min_rank"]
            mx = st["max_rank"]
            bs = st["bucket_size"]
            nb = st["num_buckets"]

            r_low = blk["r_low"]
            r_high = blk["r_high"]

            # clamp
            if r_high < mn or r_low > mx:
                continue
            b_start = max(0, (max(r_low, mn) - mn) // bs)
            b_end   = min(nb - 1, (min(r_high, mx) - mn) // bs)

            affected = list(range(int(b_start), int(b_end) + 1))
            if affected:
                reward_per_bucket = per_block_reward / float(len(affected))
                for b in affected:
                    st["bucket_fw_gain"][b] += reward_per_bucket
                    st["bucket_counts"][b]  += 1

            # update good/cool
            if per_block_reward > 1e-9:
                st["good_intervals"].append((r_low, r_high, per_block_reward, batch_idx))
            elif per_block_reward < COOL_MIN_DELTA_FW:
                st["cool_intervals"].append((r_low, r_high, batch_idx))

        # -------------- important batch summary (always log) --------------
        _log_important(
            f"Batch {batch_idx}: blocks={len(batch_blocks)}, "
            f"BW {bw_old:.6f} -> {bw_current:.6f} (ΔBW={bw_current - bw_old:+.6f}), "
            f"deltaFW={delta_fw:+.6f}, "
            f"F→B_wt={sum_f2b:.6f}, B→F_wt={sum_b2f:.6f}, "
            f"new_F→B_edges={added_sz}, total_F→B_edges={len(f2b_edges_global)}, "
            f"wall={batch_elapsed:.2f}s"
        )

        # -------------- periodic save --------------
        if save_every is not None and save_every > 0 and (batch_idx % save_every == 0):
            rows = [(index_to_node[idx], current_scores[idx]) for idx in current_scores]
            rows_sorted = sorted(rows, key=lambda x: x[1])
            pd.DataFrame(rows_sorted, columns=["Node ID", "Order"]).to_csv(output_path, index=False)
            _log_important(f"Saved intermediate ranking at batch {batch_idx} -> {output_path}")

        # -------------- periodic verify (full recompute BW) --------------
        if verify_every is not None and verify_every > 0 and (batch_idx % verify_every == 0):
            fw_manual = float(compute_forward_weight(edges_indexed, current_scores))
            bw_manual = float(total_weight - fw_manual)
            if abs(bw_manual - bw_current) > 1e-6:
                raise RuntimeError(
                    f"BW mismatch at verification batch {batch_idx}: tracked={bw_current:.6f}, manual={bw_manual:.6f}"
                )
            _log_important(f"Verify batch {batch_idx}: BW tracked={bw_current:.6f}, manual={bw_manual:.6f}")

    # ----------------- final save -----------------
    rows = [(index_to_node[idx], current_scores[idx]) for idx in current_scores]
    rows_sorted = sorted(rows, key=lambda x: x[1])
    pd.DataFrame(rows_sorted, columns=["Node ID", "Order"]).to_csv(output_path, index=False)

    elapsed = time.time() - t_start
    _log_important(
        f"Done: batches={batch_idx}, elapsed={elapsed:.2f}s, "
        f"final_BW={bw_current:.6f}, total_unique_F→B_edges={len(f2b_edges_global)}, "
        f"ranking_written={output_path}"
    )

    return current_scores, bw_history, f2b_edges_global, output_path



In [12]:
def compute_direction_change_stats(
    edges,
    before_scores,
    after_scores,
    restrict_to_edges=None,
):
    """
    Compute how many edges changed direction between two rankings and
    the total weight of those changes.

    IMPORTANT (consistency with BW tracking):
      - wt_B2F is the amount of weight REMOVED from backward edges (BW decreases by wt_B2F)
      - wt_F2B is the amount of weight ADDED to backward edges (BW increases by wt_F2B)
      - Therefore, net BW change = (+wt_F2B) - (wt_B2F)

    Parameters
    ----------
    edges : iterable of (u, v, w)
        All directed edges with weights.
    before_scores : dict
        node -> rank BEFORE the phase.
    after_scores : dict
        node -> rank AFTER the phase.
    restrict_to_edges : iterable of (u, v) or None
        If not None, only consider edges whose (u, v) is in this set.

    Returns
    -------
    num_B2F : int
        Number of edges that went from backward to forward.
    wt_B2F : float
        Total weight of edges that went from backward to forward. (BW decrease)
    num_F2B : int
        Number of edges that went from forward to backward.
    wt_F2B : float
        Total weight of edges that went from forward to backward. (BW increase)
    """
    restrict_set = set(restrict_to_edges) if restrict_to_edges is not None else None

    num_B2F = 0
    wt_B2F = 0.0
    num_F2B = 0
    wt_F2B = 0.0

    for (u, v, w) in edges:
        if restrict_set is not None and (u, v) not in restrict_set:
            continue

        bu = before_scores[u]
        bv = before_scores[v]
        au = after_scores[u]
        av = after_scores[v]

        # Orientation before/after (ties ignored)
        before_forward = (bu < bv)
        before_backward = (bu > bv)
        after_forward = (au < av)
        after_backward = (au > av)

        if before_backward and after_forward:
            num_B2F += 1
            wt_B2F += float(w)
        elif before_forward and after_backward:
            num_F2B += 1
            wt_F2B += float(w)

    return num_B2F, wt_B2F, num_F2B, wt_F2B


In [13]:
def hybrid_refine_func1_func2(
    csv_path,
    initial_ranking_path,
    output_ranking_path,        # single CSV that is always overwritten
    log_path,                   # text log file (explicit)
    h_hours: float = 4.0,
    func2_f2b_limit: int = 200,
    func1_full_every: int = 50,
    func1_log_every_round: int = 1,
    func1_bw_check_every_round: int = 5,   # RENAMED: BW not FW
    func2_verify_every: int = 5,
    c: float = 1.0,             # kept for compatibility, IGNORED
):
    """
    Hybrid process with a SINGLE ranking CSV.

    Reports and optimizes BACKWARD WEIGHT (BW) only (goal: minimize BW).
    Uses ONLY log_message(...) for logging (no prints, no custom log writers).
    """

    import os
    import time

    # ------------------------------
    # Load graph once for statistics
    # ------------------------------
    edges, node_to_index, index_to_node = read_graph(csv_path)

    # Total weight (constant): lets us compute BW = total - FW when convenient
    total_weight = 0.0
    for (_, _, w) in edges:
        total_weight += float(w)

    # ------------------------------
    # Ensure output directory exists
    # ------------------------------
    out_dir = os.path.dirname(output_ranking_path) or "."
    os.makedirs(out_dir, exist_ok=True)

    # ------------------------------
    # Processor counts (for log)
    # ------------------------------
    try:
        func2_procs_aff = len(os.sched_getaffinity(0))
    except Exception:
        func2_procs_aff = None

    func1_procs = os.cpu_count() or 1
    func2_procs = func2_procs_aff if func2_procs_aff is not None else func1_procs

    # ------------------------------
    # Write header in log file
    # ------------------------------
    # We avoid direct file writes and use the required logger.
    # (This will append; if you want a fresh file each run, delete it outside or add an explicit truncate elsewhere.)
    log_message("# HYBRID func1/func2 LOG (all rankings in ONE CSV)", log_path)
    log_message(f"# ranking_csv = {output_ranking_path}", log_path)
    log_message("# Per-PHASE statistics:", log_path)
    log_message("#   - BW (Backward Weight) before/after each phase", log_path)
    log_message("#   - Counts & total weight of edges that changed direction", log_path)
    log_message(f"# func1_processors (dynamic round) = {func1_procs}", log_path)
    log_message(f"# func2_processors (largest SCC intervals) = {func2_procs}", log_path)
    log_message("# COLUMNS:", log_path)
    log_message("# phase_idx, phase_type, duration_sec, bw_before, bw_after, num_B2F, wt_B2F, num_F2B, wt_F2B, extra", log_path)
    log_message("# NOTE: parameter c is ignored; initial func1 uses ALL backward edges.", log_path)

    # ------------------------------
    # Time limit
    # ------------------------------
    start_ts = time.time()
    deadline = start_ts + h_hours * 3600.0

    phase_idx = 0
    func1_runs = 0
    func2_runs = 0

    # Global pool of unassessed edges from func1 phases, as (u,v) pairs
    unassessed_edges_global = set()

    # ------------------------------
    # Helper: load scores + BW from a ranking CSV
    # ------------------------------
    def bw_from_ranking_path(ranking_path: str):
        scores = load_initial_scores(ranking_path, node_to_index)

        # Efficient BW computation via total - FW (your requested trick)
        fw = compute_forward_weight(edges, scores)
        bw = total_weight - float(fw)
        return scores, bw

    current_input_path = initial_ranking_path

    # =====================================================
    # PHASE 1: func1 on ALL backward edges (edge_subset=None)
    # =====================================================
    if time.time() >= deadline:
        log_message("No time left to start initial func1; exiting.", log_path)
        return output_ranking_path

    phase_idx += 1
    func1_runs += 1
    phase_type = "func1_initial_global"

    before_scores, bw_before = bw_from_ranking_path(current_input_path)

    t0 = time.time()
    remaining_h = max(0.0, (deadline - t0) / 3600.0)

    # Updated signature of refine_ranking_parallel_dynamic includes log_path and returns BW
    _, _, remaining_pairs_unassessed = refine_ranking_parallel_dynamic(
        csv_path=csv_path,
        initial_ranking_path=current_input_path,
        output_excel=output_ranking_path,
        log_path=log_path,
        MAX_HOURS=remaining_h,
        LOG_EVERY_ROUND=func1_log_every_round,
        BW_CHECK_EVERY_ROUND=func1_bw_check_every_round,
        edge_subset=None,  # ALL backward edges
    )
    t1 = time.time()

    after_scores, bw_after = bw_from_ranking_path(output_ranking_path)

    num_B2F, wt_B2F, num_F2B, wt_F2B = compute_direction_change_stats(
        edges, before_scores, after_scores
    )

    unassessed_edges_global = set(remaining_pairs_unassessed or [])
    extra_subset_info = f"initial_global_func1, unassessed_edges={len(unassessed_edges_global)}"

    log_message(
        f"{phase_idx}, {phase_type}, {t1 - t0:.2f}, "
        f"{bw_before:.2f}, {bw_after:.2f}, "
        f"{num_B2F}, {wt_B2F:.2f}, {num_F2B}, {wt_F2B:.2f}, "
        f"{extra_subset_info}",
        log_path
    )

    current_input_path = output_ranking_path

    # =====================================================
    # MAIN LOOP: func2 ↔ func1 until time runs out
    # =====================================================
    last_f2b_edges = None

    while True:
        # ------------------------------------
        # (A) func2
        # ------------------------------------
        if time.time() >= deadline:
            log_message("Time limit reached before starting next func2; stopping.", log_path)
            break

        phase_idx += 1
        func2_runs += 1
        phase_type = f"func2_run{func2_runs}"

        before_scores, bw_before = bw_from_ranking_path(current_input_path)

        t0 = time.time()
        scores_after_func2, fw_history, f2b_edges_global, out_path_used = parallel_refine_largest_scc_intervals(
            max_backward_flips=func2_f2b_limit,
            verify_every=func2_verify_every,
            csv_path=csv_path,
            initial_ranking_path=current_input_path,
            output_path=output_ranking_path,  # always overwrite the same file
            debug=False,
            log_path=log_path,                # IMPORTANT: pass log_path if supported
        )
        t1 = time.time()

        after_scores, bw_after = bw_from_ranking_path(output_ranking_path)

        num_B2F, wt_B2F, num_F2B, wt_F2B = compute_direction_change_stats(
            edges, before_scores, after_scores
        )

        last_f2b_edges = f2b_edges_global or []

        extra = f"func2_max_F2B={func2_f2b_limit}, func2_actual_F2B={len(last_f2b_edges)}"
        log_message(
            f"{phase_idx}, {phase_type}, {t1 - t0:.2f}, "
            f"{bw_before:.2f}, {bw_after:.2f}, "
            f"{num_B2F}, {wt_B2F:.2f}, {num_F2B}, {wt_F2B:.2f}, "
            f"{extra}",
            log_path
        )

        current_input_path = output_ranking_path

        if not last_f2b_edges and not unassessed_edges_global:
            log_message("func2 produced 0 F->B edges and no unassessed edges remain; stopping.", log_path)
            break

        # ------------------------------------
        # (B) func1: on (F2B from func2) ∪ (unassessed so far),
        #     BUT if unassessed is EMPTY, run GLOBAL func1.
        # ------------------------------------
        if time.time() >= deadline:
            log_message("Time limit reached before starting func1-after-func2; stopping.", log_path)
            break

        f2b_pairs = set()
        for e in last_f2b_edges:
            if isinstance(e, tuple) and len(e) >= 2:
                f2b_pairs.add((e[0], e[1]))

        unassessed_before = len(unassessed_edges_global)

        if unassessed_before == 0:
            use_global_for_this_func1 = True
            subset_for_func1 = None
        else:
            use_global_for_this_func1 = False
            subset_for_func1 = set(unassessed_edges_global)
            subset_for_func1.update(f2b_pairs)

        if (not use_global_for_this_func1) and (not subset_for_func1):
            log_message("No edges to refine in func1-after-func2 (subset empty). Skipping func1.", log_path)
            continue

        phase_idx += 1
        func1_runs += 1
        phase_type = f"func1_after_func2_run{func1_runs}"

        before_scores, bw_before = bw_from_ranking_path(current_input_path)

        t0 = time.time()
        remaining_h = max(0.0, (deadline - t0) / 3600.0)

        _, _, remaining_pairs_unassessed = refine_ranking_parallel_dynamic(
            csv_path=csv_path,
            initial_ranking_path=current_input_path,
            output_excel=output_ranking_path,
            log_path=log_path,
            MAX_HOURS=remaining_h,
            LOG_EVERY_ROUND=func1_log_every_round,
            BW_CHECK_EVERY_ROUND=func1_bw_check_every_round,
            edge_subset=None if use_global_for_this_func1 else list(subset_for_func1),
        )
        t1 = time.time()

        after_scores, bw_after = bw_from_ranking_path(output_ranking_path)

        num_B2F, wt_B2F, num_F2B, wt_F2B = compute_direction_change_stats(
            edges, before_scores, after_scores
        )

        # Stats restricted to those F→B edges from func2
        num_B2F_sub, wt_B2F_sub, num_F2B_sub, wt_F2B_sub = compute_direction_change_stats(
            edges, before_scores, after_scores, restrict_to_edges=list(f2b_pairs)
        )

        unassessed_edges_global = set(remaining_pairs_unassessed or [])
        unassessed_after = len(unassessed_edges_global)

        subset_size_str = "ALL_BACKWARD" if use_global_for_this_func1 else str(len(subset_for_func1))

        extra = (
            f"mode={'global' if use_global_for_this_func1 else 'targeted'}, "
            f"subset_total={subset_size_str}, "
            f"subset_F2B_size={len(f2b_pairs)}, "
            f"subset_B2F={num_B2F_sub}, subset_wt_B2F={wt_B2F_sub:.2f}, "
            f"subset_F2B={num_F2B_sub}, subset_wt_F2B={wt_F2B_sub:.2f}, "
            f"unassessed_before={unassessed_before}, "
            f"unassessed_after_func1={unassessed_after}"
        )

        log_message(
            f"{phase_idx}, {phase_type}, {t1 - t0:.2f}, "
            f"{bw_before:.2f}, {bw_after:.2f}, "
            f"{num_B2F}, {wt_B2F:.2f}, {num_F2B}, {wt_F2B:.2f}, "
            f"{extra}",
            log_path
        )

        current_input_path = output_ranking_path

        # ------------------------------------
        # (C) every func1_full_every: extra global func1
        # ------------------------------------
        if func1_full_every > 0 and (func1_runs % func1_full_every == 0):
            if time.time() >= deadline:
                log_message("Time limit reached before extra-every-N func1; stopping.", log_path)
                break

            phase_idx += 1
            phase_type = f"func1_every{func1_full_every}_global"

            before_scores, bw_before = bw_from_ranking_path(current_input_path)

            t0 = time.time()
            remaining_h = max(0.0, (deadline - t0) / 3600.0)

            _, _, remaining_pairs_unassessed = refine_ranking_parallel_dynamic(
                csv_path=csv_path,
                initial_ranking_path=current_input_path,
                output_excel=output_ranking_path,
                log_path=log_path,
                MAX_HOURS=remaining_h,
                LOG_EVERY_ROUND=func1_log_every_round,
                BW_CHECK_EVERY_ROUND=func1_bw_check_every_round,
                edge_subset=None,
            )
            t1 = time.time()

            after_scores, bw_after = bw_from_ranking_path(output_ranking_path)

            num_B2F, wt_B2F, num_F2B, wt_F2B = compute_direction_change_stats(
                edges, before_scores, after_scores
            )

            unassessed_edges_global = set(remaining_pairs_unassessed or [])

            bw_delta = bw_after - bw_before  # negative is good

            log_message(
                f"{phase_idx}, {phase_type}, {t1 - t0:.2f}, "
                f"{bw_before:.2f}, {bw_after:.2f}, "
                f"{num_B2F}, {wt_B2F:.2f}, {num_F2B}, {wt_F2B:.2f}, "
                f"note=every_{func1_full_every}, unassessed={len(unassessed_edges_global)}, dBW={bw_delta:+.2f}",
                log_path
            )

            current_input_path = output_ranking_path

    log_message(
        f"Hybrid process finished. Total func1_runs={func1_runs}, func2_runs={func2_runs}, "
        f"final_unassessed_edges={len(unassessed_edges_global)}",
        log_path
    )
    return output_ranking_path


In [14]:
def compute_backward_weight(edges, scores):
    """
    Backward Weight (BW): total weight of edges (u->v) that go backward w.r.t. scores,
    i.e., score[u] > score[v]. (Scores are unique in your pipeline.)
    """
    bw = 0.0
    for (u, v, w) in edges:
        if scores[u] > scores[v]:
            bw += float(w)
    return bw


In [15]:
def generate_scc_ranking(edges, index_to_node, log_path):
    """
    Generates a ranking based on the topological order of Strongly Connected Components (SCCs).

    Input
    -----
    edges : list of (u_idx, v_idx, weight)
        Directed weighted edges (weight is ignored for SCC structure).
    index_to_node : dict
        Maps internal index -> original node ID/name.
    log_path : str
        Path for log_message.

    Output
    ------
    dict {original_node_id/name: rank}
        A total order produced by ordering SCCs topologically, then listing members within each SCC.
        (Order inside an SCC is arbitrary but deterministic here.)
    """
    import networkx as nx

    log_message("generate_scc_ranking: building DiGraph from edges...", log_path)

    G = nx.DiGraph()
    for u, v, _w in edges:
        G.add_edge(u, v)

    # Ensure all nodes exist (including isolated nodes)
    for idx in index_to_node.keys():
        if idx not in G:
            G.add_node(idx)

    log_message("generate_scc_ranking: condensing graph into SCC DAG...", log_path)
    scc_graph = nx.condensation(G)

    log_message("generate_scc_ranking: topological sorting SCC DAG...", log_path)
    scc_topo_order = list(nx.topological_sort(scc_graph))

    log_message("generate_scc_ranking: assigning ranks to nodes...", log_path)
    node_ranks = {}
    current_rank = 0

    # Determinism: sort members so output is stable across runs
    for scc_id in scc_topo_order:
        members = scc_graph.nodes[scc_id].get("members", [])
        for member_idx in sorted(members):
            original_node_name = index_to_node[member_idx]
            node_ranks[original_node_name] = current_rank
            current_rank += 1

    log_message(
        f"generate_scc_ranking: done. n_nodes_ranked={len(node_ranks)}, n_scc={len(scc_topo_order)}",
        log_path
    )
    return node_ranks

def compare_forward_weights(edges, scores_before, scores_after):
    """
    Compare edge orientations between two rankings.

    Returns:
      sum_sbef_only:    total weight of edges that are forward in BEFORE but NOT forward in AFTER
                        (i.e., forward->backward OR forward->tie)
      sum_saft_only:    total weight of edges that are forward in AFTER but NOT forward in BEFORE
                        (i.e., backward->forward OR tie->forward)
      sum_both_forward: total weight of edges forward in BOTH
      sum_both_backward:total weight of edges backward in BOTH
      only_before_global: set of (u, v) edges that went forward->backward (strictly)
    """
    sum_sbef_only = 0.0
    sum_saft_only = 0.0
    sum_both_forward = 0.0
    sum_both_backward = 0.0
    only_before_global = set()

    for (u, v, w) in edges:
        bu = scores_before[u]
        bv = scores_before[v]
        au = scores_after[u]
        av = scores_after[v]

        before_forward = (bu < bv)
        before_backward = (bu > bv)

        after_forward = (au < av)
        after_backward = (au > av)

        # forward in BOTH
        if before_forward and after_forward:
            sum_both_forward += float(w)

        # backward in BOTH
        if before_backward and after_backward:
            sum_both_backward += float(w)

        # forward only in BEFORE (not forward after)
        if before_forward and (not after_forward):
            sum_sbef_only += float(w)
            # specifically forward->backward (strict), collect as F2B edge
            if after_backward:
                only_before_global.add((u, v))

        # forward only in AFTER (not forward before)
        if after_forward and (not before_forward):
            sum_saft_only += float(w)

    return sum_sbef_only, sum_saft_only, sum_both_forward, sum_both_backward, only_before_global


In [16]:
# 1) Define Base Directory and Input File
import os
import datetime
import pandas as pd

base_dir = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/"
graph_file_path = os.path.join(base_dir, "mm9a.d")

# IMPORTANT: set an explicit log path for ALL logs in this pipeline
log_path = os.path.join(base_dir, "mm9a_dimacs_hybrid_phases_log-48cpu.txt")


# 2) Read Graph (log via log_message only)
log_message(f"Reading graph from {graph_file_path} ...", log_path)

# Note: read_graph returns (edges_indexed, node_to_index, index_to_node)
edges, node_to_index, index_to_node = read_graph(graph_file_path)

log_message(
    f"Graph loaded: n_nodes={len(node_to_index)}, n_edges={len(edges)}",
    log_path
)


# 3) Generate SCC-based Ranking (log via log_message only)
scc_ranking_dict = generate_scc_ranking(edges, index_to_node, log_path)


# 4) Save SCC Ranking to CSV (no prints; log via log_message only)
# We match the columns expected by load_initial_scores: "Node ID" and "Order"
initial_scc_csv_path = os.path.join(base_dir, "mm9a_dimacs_scc_init.csv")

data_rows = [{"Node ID": node_name, "Order": int(score)} for node_name, score in scc_ranking_dict.items()]
pd.DataFrame(data_rows).to_csv(initial_scc_csv_path, index=False)

log_message(f"Generated SCC-based initial ranking at: {initial_scc_csv_path}", log_path)


# 5) Call the Hybrid Function (single output CSV always overwritten)
final_ranking_path = hybrid_refine_func1_func2(
    csv_path=graph_file_path,
    initial_ranking_path=initial_scc_csv_path,  # SCC init ranking
    output_ranking_path=os.path.join(base_dir, "mm9a_dimacs_hybrid_ranking-48cpu.csv"),
    log_path=log_path,
    h_hours=72.0,
    c=1.0,  # ignored (kept for compatibility)
)

log_message(f"Hybrid finished. Final ranking path: {final_ranking_path}", log_path)


EmptyDataError: No columns to parse from file